In [13]:

import matplotlib.pyplot as plt
import matplotlib.tri as mtri

import math
from typing import Dict, List, Tuple, Literal,Optional
import numpy as np
#NeighborMode = Literal["Edges", "Extented_edges"]

# mesh 생성 코드
def mesh(
    x_s: float, x_f: float,
    y_s: float, y_f: float,
    x_c: float, y_c: float,
    r: float, mod: str

) -> Tuple[np.ndarray, Dict[Tuple[int, int], int]]:
    """
    Equilateral triangular mesh (side length r) inside rectangle [x_s,x_f] x [y_s,y_f],
    anchored at (x_c, y_c).

    Returns:
      vertices: (N,2) float array
      idx_map: dict mapping (i,j) -> vertex_index (lattice coords)
    """
    if r <= 0:
        raise ValueError("r must be positive.")
    if x_s > x_f:
        x_s, x_f = x_f, x_s
    if y_s > y_f:
        y_s, y_f = y_f, y_s
    mod=mod.strip()
    if mod == "Square":
        i_min = math.ceil((x_s - x_c) / r)
        i_max = math.floor((x_f - x_c) / r)
        j_min = math.ceil((y_s - y_c) / r)
        j_max = math.floor((y_f - y_c) / r)

        vertices: List[Tuple[float, float]] = []
        idx_map: Dict[Tuple[int, int], int] = {}

        for j in range(j_min, j_max + 1):
            y = y_c + j * r
            for i in range(i_min, i_max + 1):
                x = x_c + i * r
                # boundary included
                if (x_s <= x <= x_f) and (y_s <= y <= y_f):
                    idx_map[(i, j)] = len(vertices)
                    vertices.append((x, y))

        return np.array(vertices, dtype=float), idx_map
    if mod=='Hexa':
        dy = r * math.sqrt(3) / 2.0  # row spacing

        j_min = math.ceil((y_s - y_c ) / dy)
        j_max = math.floor((y_f - y_c ) / dy)

        vertices: List[Tuple[float, float]] = []
        idx_map: Dict[Tuple[int, int], int] = {}

        for j in range(j_min, j_max + 1):
            y = y_c + j * dy

            # x = x_c + i*r + j*(r/2)
            i_min = math.ceil((x_s - x_c - (j * r / 2.0)) / r)
            i_max = math.floor((x_f - x_c - (j * r / 2.0) ) / r)

            for i in range(i_min, i_max + 1):
                x = x_c + i * r + j * (r / 2.0)
                if (x_s) <= x <= (x_f) and (y_s) <= y <= (y_f):
                    idx_map[(i, j)] = len(vertices)
                    vertices.append((x, y))
        return np.array(vertices, dtype=float),  idx_map


def vertex_index_from_xy(
    x: float, y: float,
    idx_map: Dict[Tuple[int, int], int],
    mod: Literal["Square", "Hexa"],
    x_c: float, y_c: float,
    r: float,
) -> int:
    """
    Return vertex index for the nearest lattice vertex to (x,y).
    Uses idx_map from mesh().

    Square:
      i = round((x-x_c)/r), j = round((y-y_c)/r)

    Hexa:
      dy = r*sqrt(3)/2
      j = round((y-y_c)/dy)
      i = round((x-x_c - j*(r/2))/r)
    """
    mod = mod.strip()
    if r <= 0:
        raise ValueError("r must be positive.")

    if mod == "Square":
        i = int(round((x - x_c) / r))
        j = int(round((y - y_c) / r))
    elif mod == "Hexa":
        dy = r * math.sqrt(3) / 2.0
        j = int(round((y - y_c) / dy))
        i = int(round((x - x_c - j * (r / 2.0)) / r))
    else:
        raise ValueError("mod must be 'Square' or 'Hexa'.")

    key = (i, j)
    if key not in idx_map:
        raise KeyError(f"(x,y)=({x},{y}) -> (i,j)=({i},{j}) not in idx_map (outside domain?)")
    return idx_map[key]

mesh_mode = Literal["Square", "Hexa"]
NeighborMode = Literal["Edges_only", "Extended_edges","Extra_extended_edges"]
#인접한 mesh 분석
def build_vertex_adjacency(
    idx_map: Dict[Tuple[int, int], int],
    num_vertices: int,
    mod: mesh_mode = "Hexa",
    mode: NeighborMode = "Extended_edges",
    *,
    missing: int = -1,   # 없는 이웃 채우는 값
) -> Tuple[np.ndarray, List[Tuple[int, int]]]:
    """
    Returns:
      adj: (N, K) int array, adj[v, k] = neighbor index or 'missing'
      offsets: length K, adjacency index k가 의미하는 (di, dj)
    """
    mod = mod.strip()
    mode = mode.strip()

    # ---- offsets depending on mesh type ----
    if mod == "Square":
        offsets = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        if mode in ("Extended_edges", "Extra_extended_edges"):
            offsets = [
                (0, 1), (1, 1), (1, 0), (1, -1),
                (0, -1), (-1, -1), (-1, 0), (-1, 1),
            ]
        if mode == "Extra_extended_edges":
            offsets = [
                (0, 1), (1, 2), (1, 1), (2, 1),
                (1, 0), (2, -1), (1, -1), (1, -2),
                (0, -1), (-1, -2), (-1, -1), (-2, -1),
                (-1, 0), (-2, 1), (-1, 1), (-1, 2),
            ]

    elif mod == "Hexa":
        offsets = [(0,1),(1,0),(1,-1),(0,-1),(-1,0),(-1,1)]
        if mode in ("Extended_edges", "Extra_extended_edges"):
            offsets = [
                (-1,2),(0,1),(1,1),(1,0),(2,-1),(1,-1),
                (1,-2),(0,-1),(-1,-1),(-1,0),(-2,1),(-1,1)
            ]
        if mode == "Extra_extended_edges":
            offsets = [
                (-1,2),(-1,3),(0,1),(1,2),(1,1),(2,1),
                (1,0),(3,-1),(2,-1),(3,-2),(1,-1),(2,-3),
                (1,-2),(1,-3),(0,-1),(-1,-2),(-1,-1),(-2,-1),
                (-1,0),(-3,1),(-2,1),(-3,2),(-1,1),(-2,3)
            ]
    else:
        raise ValueError("mod must be 'Square' or 'Hexa'.")

    K = len(offsets)
    adj = np.full((num_vertices, K), missing, dtype=np.int32)

    # idx_map의 (i,j)마다 K개 슬롯을 채운다
    for (i, j), v in idx_map.items():
        for k, (di, dj) in enumerate(offsets):
            u = idx_map.get((i + di, j + dj))
            if u is not None:
                adj[v, k] = u

    return adj
def angle_from_y_clockwise_deg(dx: float, dy: float) -> float:
    """0°=+y, 90°=+x, 180°=-y, 270°=-x"""
    ang = math.degrees(math.atan2(dx, dy))
    return (ang + 360.0) % 360.0

def build_adjacency_angles(
    vertices: np.ndarray,     # (N,2)
    adj_fixed: np.ndarray,    # (N,K)  (-1 for missing)
    *,
    missing_angle: float = float("nan"),
) -> np.ndarray:
    """
    Returns:
      ang: (N,K) float array, ang[v,k] = angle_deg or NaN if missing neighbor
    """
    N, K = adj_fixed.shape
    ang = np.full((N, K), missing_angle, dtype=np.float64)

    for v in range(N):
        xv, yv = vertices[v]
        for k in range(K):
            u = int(adj_fixed[v, k])
            if u < 0:
                continue
            xu, yu = vertices[u]
            dx, dy = (xu - xv), (yu - yv)
            ang[v, k] = angle_from_y_clockwise_deg(dx, dy)

    return ang

def fixed_to_ragged(adj_fixed: np.ndarray, missing: int = -1) -> List[List[int]]:
    return [[int(u) for u in row if int(u) != missing] for row in adj_fixed]
from dataclasses import dataclass

@dataclass
class Vortex:
    x0: float
    y0: float
    Gamma: float       # circulation strength
    core: float = 1.0  # core radius a (regularization)

def angle_from_y_clockwise_deg_vortex(ux: np.ndarray, uy: np.ndarray) -> np.ndarray:
    """
    Angle from +y axis, clockwise, in degrees.
    0°=+y, 90°=+x, 180°=-y, 270°=-x
    """
    ang = np.degrees(np.arctan2(ux, uy))  # <-- IMPORTANT: atan2(ux, uy)
    return (ang + 360.0) % 360.0

def vortex_velocity(points: np.ndarray, vortex: Vortex, eps: float = 1e-12) -> Tuple[np.ndarray, np.ndarray]:
    """
    points: (N,2) array [[x,y],...]
    returns: ux, uy each (N,)
    Lamb-Oseen-like regularized point vortex
    """
    x = points[:, 0]
    y = points[:, 1]
    dx = x - vortex.x0
    dy = y - vortex.y0
    r2 = dx*dx + dy*dy

    # regularization factor alpha(r) = 1 - exp(-r^2/a^2)
    a2 = max(vortex.core, eps) ** 2
    alpha = 1.0 - np.exp(-r2 / a2)

    # avoid divide-by-zero at center
    denom = np.maximum(r2, eps)

    coef = vortex.Gamma / (2.0 * math.pi)
    ux = -coef * (dy / denom) * alpha
    uy =  coef * (dx / denom) * alpha
    return ux, uy

def composite_current(
    points: np.ndarray,
    vortices: List[Vortex],
    uniform: Tuple[float, float] = (0.0, 0.0),
    return_per_vortex: bool = False
) -> Dict[str, np.ndarray]:
    """
    returns dict with:
      total_ux, total_uy, speed, angle_deg
      (optional) per_vortex_ux, per_vortex_uy, per_vortex_speed, per_vortex_angle_deg
    """
    N = points.shape[0]
    total_ux = np.full(N, uniform[0], dtype=float)
    total_uy = np.full(N, uniform[1], dtype=float)

    if return_per_vortex:
        K = len(vortices)
        per_ux = np.zeros((K, N), dtype=float)
        per_uy = np.zeros((K, N), dtype=float)

    for k, vtx in enumerate(vortices):
        ux, uy = vortex_velocity(points, vtx)
        total_ux += ux
        total_uy += uy
        if return_per_vortex:
            per_ux[k] = ux
            per_uy[k] = uy

    speed = np.hypot(total_ux, total_uy)
    angle_deg = angle_from_y_clockwise_deg_vortex(total_ux, total_uy)

    out = {
        "total_ux": total_ux,
        "total_uy": total_uy,
        "speed": speed,
        "angle_deg": angle_deg,
    }
    return out
def Gamma_from_RV(R: float, V: float) -> float:
    return 2.0 * math.pi * R * V

def _rand_around(
    rng: np.random.Generator,
    base: float,
    rel_std: float,
    clip_lo: float,
    clip_hi: float
) -> float:
    """base 주변 랜덤(정규) 변동 + 클리핑."""
    v = base * (1.0 + rng.normal(0.0, rel_std))
    return float(np.clip(v, clip_lo * base, clip_hi * base))


def _sample_pos(
    rng: np.random.Generator,
    Lx: float,
    Ly: float,
    margin: float,
    existing_xy: List[Tuple[float, float]],
    min_dist: float,
    max_tries: int = 5000,
) -> Tuple[float, float]:
    """영역 내 좌표 샘플링 + 기존 점들과 최소거리 제약."""
    for _ in range(max_tries):
        x0 = float(rng.uniform(margin, Lx - margin))
        y0 = float(rng.uniform(margin, Ly - margin))
        ok = True
        for ex, ey in existing_xy:
            if (x0 - ex) ** 2 + (y0 - ey) ** 2 < min_dist ** 2:
                ok = False
                break
        if ok:
            return x0, y0

    # 제약이 빡세서 실패할 수 있음. 마지막 시도값 반환(원하면 raise로 변경 가능)
    return x0, y0
def _rotate_points_2d(points: np.ndarray, angle_rad: float) -> np.ndarray:
    c = math.cos(angle_rad)
    s = math.sin(angle_rad)
    R = np.array([[c, -s],
                  [s,  c]], dtype=float)
    return points @ R.T
def make_random_vortices(
    *,
    Lx: float = 3600.0,
    Ly: float = 3600.0,
    seed: Optional[int] = None,

    # 개수
    n_mid: int = 3,
    n_small: int = 6,

    # 강도 스케일
    big_R: float = 1800.0,
    big_V: float = 0.5,
    mid_R: float = 900.0,
    mid_V: float = 0.40,
    small_R: float = 250.0,
    small_V: float = 0.4,

    # 코어 반경
    big_core: float = 900.0,
    mid_core: float = 550.0,
    small_core: float = 350.0,

    # 랜덤 변동 폭
    gamma_rel_std_big: float = 0.06,
    gamma_rel_std_mid: float = 0.08,
    gamma_rel_std_small: float = 0.10,
    core_rel_std: float = 0.08,

    # 배치 제약
    margin: float = 120.0,
    min_dist_big: float = 900.0,
    min_dist_mid: float = 450.0,
    min_dist_small: float = 250.0,

    # big template 크기
    pair_dist_base_frac: float = 0.50,   # pair 거리의 기준: min(Lx, Ly)의 비율
    pair_dist_rel_std: float = 0.12,
    tri_scale_base_frac: float = 0.42,   # triangle 크기 기준
    tri_scale_rel_std: float = 0.12,

    # big 중심점 흔들기
    big_center_jitter_frac_x: float = 0.08,
    big_center_jitter_frac_y: float = 0.08,

    # big 회전 방향 / 부호 랜덤성
    randomize_big_signs: bool = True,
    rebalance_big_signs: bool = True,

    # mid / small 부호 랜덤성
    randomize_mid_signs: bool = True,
    randomize_small_signs: bool = True,
    rebalance_mid_small_signs: bool = True,

    # uniform flow 랜덤성
    randomize_uniform: bool = True,
    uniform_speed_min: float = 0.0,
    uniform_speed_max: float = 0.08,

    # fallback
    default_uniform: Tuple[float, float] = (0.0, 0.0),
):
    """
    Returns:
      vortices: List[Vortex]
      uniform:  (Ux, Uy)

    특징:
      - big vortices는 2개 또는 3개
      - 2개면 pair 구조, 3개면 triangle 구조
      - 단, pair / triangle 전체 orientation은 랜덤 회전
      - big vortex의 부호(회전 방향)도 랜덤
      - mid / small도 독립적인 랜덤 부호
      - uniform flow도 방향 및 세기 랜덤
    """
    rng = np.random.default_rng(seed)

    # --------------------------------------------------
    # base Gamma
    # --------------------------------------------------
    Gamma_big_base = Gamma_from_RV(big_R, big_V)
    Gamma_mid_base = Gamma_from_RV(mid_R, mid_V)
    Gamma_small_base = Gamma_from_RV(small_R, small_V)

    vortices: List[Vortex] = []
    existing_xy: List[Tuple[float, float]] = []

    # --------------------------------------------------
    # local helper: boundary 안으로 shift
    # --------------------------------------------------
    def _shift_points_into_domain(pts: np.ndarray, margin_local: float) -> np.ndarray:
        pts = pts.copy()

        min_x, max_x = np.min(pts[:, 0]), np.max(pts[:, 0])
        min_y, max_y = np.min(pts[:, 1]), np.max(pts[:, 1])

        shift_x = 0.0
        shift_y = 0.0

        if min_x < margin_local:
            shift_x += (margin_local - min_x)
        if max_x > Lx - margin_local:
            shift_x -= (max_x - (Lx - margin_local))
        if min_y < margin_local:
            shift_y += (margin_local - min_y)
        if max_y > Ly - margin_local:
            shift_y -= (max_y - (Ly - margin_local))

        pts[:, 0] += shift_x
        pts[:, 1] += shift_y
        return pts

    # --------------------------------------------------
    # 1) big vortices
    # --------------------------------------------------
    n_big = int(rng.integers(2, 4))   # 2 or 3

    # 중심점은 전체 domain 중심 근처에서 랜덤
    cx = float(np.clip(
        0.5 * Lx + rng.normal(0.0, big_center_jitter_frac_x * Lx),
        margin, Lx - margin
    ))
    cy = float(np.clip(
        0.5 * Ly + rng.normal(0.0, big_center_jitter_frac_y * Ly),
        margin, Ly - margin
    ))

    # orientation 랜덤
    rot = float(rng.uniform(0.0, 2.0 * math.pi))

    if n_big == 2:
        # -------------------------
        # pair template
        # -------------------------
        pair_dist_base = max(min_dist_big, pair_dist_base_frac * min(Lx, Ly))
        pair_dist = _rand_around(
            rng,
            base=pair_dist_base,
            rel_std=pair_dist_rel_std,
            clip_lo=0.7,
            clip_hi=1.3,
        )
        half = 0.5 * pair_dist

        local_pts = np.array([
            [-half, 0.0],
            [ half, 0.0],
        ], dtype=float)

        pts = _rotate_points_2d(local_pts, rot)
        pts[:, 0] += cx
        pts[:, 1] += cy
        pts = _shift_points_into_domain(pts, margin)

        # big sign randomization
        if randomize_big_signs:
            if rng.random() < 0.5:
                s1, s2 = +1.0, -1.0
            else:
                s1, s2 = -1.0, +1.0
        else:
            s1, s2 = +1.0, -1.0

        G1 = _rand_around(rng, Gamma_big_base, gamma_rel_std_big, 0.6, 1.4)
        G2 = _rand_around(rng, Gamma_big_base, gamma_rel_std_big, 0.6, 1.4)
        C1 = _rand_around(rng, big_core, core_rel_std, 0.6, 1.4)
        C2 = _rand_around(rng, big_core, core_rel_std, 0.6, 1.4)

        vortices.append(Vortex(float(pts[0, 0]), float(pts[0, 1]), s1 * G1, C1))
        vortices.append(Vortex(float(pts[1, 0]), float(pts[1, 1]), s2 * G2, C2))
        existing_xy += [
            (float(pts[0, 0]), float(pts[0, 1])),
            (float(pts[1, 0]), float(pts[1, 1])),
        ]

    else:
        # -------------------------
        # triangle template
        # local: top 1개 + bottom 2개
        # 이후 통째로 회전
        # -------------------------
        tri_scale_base = max(min_dist_big, tri_scale_base_frac * min(Lx, Ly))
        tri_scale = _rand_around(
            rng,
            base=tri_scale_base,
            rel_std=tri_scale_rel_std,
            clip_lo=0.75,
            clip_hi=1.25,
        )

        h = tri_scale
        w = 0.92 * tri_scale

        local_pts = np.array([
            [ 0.0,      +0.65 * h],   # top
            [-0.5 * w,  -0.35 * h],   # bottom-left
            [+0.5 * w,  -0.35 * h],   # bottom-right
        ], dtype=float)

        pts = _rotate_points_2d(local_pts, rot)
        pts[:, 0] += cx
        pts[:, 1] += cy
        pts = _shift_points_into_domain(pts, margin)

        if randomize_big_signs:
            big_signs = rng.choice([-1.0, 1.0], size=3)
            if rebalance_big_signs and abs(np.sum(big_signs)) == 3:
                big_signs[-1] *= -1.0
        else:
            big_signs = np.array([+1.0, -1.0, +1.0], dtype=float)

        Gs = np.array([
            _rand_around(rng, Gamma_big_base, gamma_rel_std_big, 0.6, 1.4),
            _rand_around(rng, Gamma_big_base, gamma_rel_std_big, 0.6, 1.4),
            _rand_around(rng, Gamma_big_base, gamma_rel_std_big, 0.6, 1.4),
        ], dtype=float)

        Cs = np.array([
            _rand_around(rng, big_core, core_rel_std, 0.6, 1.4),
            _rand_around(rng, big_core, core_rel_std, 0.6, 1.4),
            _rand_around(rng, big_core, core_rel_std, 0.6, 1.4),
        ], dtype=float)

        for k in range(3):
            vortices.append(
                Vortex(
                    float(pts[k, 0]),
                    float(pts[k, 1]),
                    float(big_signs[k] * Gs[k]),
                    float(Cs[k]),
                )
            )
            existing_xy.append((float(pts[k, 0]), float(pts[k, 1])))

    # --------------------------------------------------
    # 2) mid vortices
    # --------------------------------------------------
    if randomize_mid_signs:
        mid_signs = rng.choice([-1.0, 1.0], size=n_mid)
        if rebalance_mid_small_signs and n_mid >= 2 and abs(np.sum(mid_signs)) == n_mid:
            mid_signs[-1] *= -1.0
    else:
        mid_signs = np.array(
            [-1.0, +1.0] * ((n_mid + 1) // 2),
            dtype=float
        )[:n_mid]

    for k in range(n_mid):
        xm, ym = _sample_pos(
            rng,
            Lx, Ly,
            margin=max(margin, 0.6 * mid_core),
            existing_xy=existing_xy,
            min_dist=min_dist_mid,
        )
        Gmid = _rand_around(rng, Gamma_mid_base, gamma_rel_std_mid, 0.6, 1.4) * float(mid_signs[k])
        Cmid = _rand_around(rng, mid_core, core_rel_std, 0.6, 1.4)
        vortices.append(Vortex(xm, ym, Gmid, Cmid))
        existing_xy.append((xm, ym))

    # --------------------------------------------------
    # 3) small vortices
    # --------------------------------------------------
    if randomize_small_signs:
        small_signs = rng.choice([-1.0, 1.0], size=n_small)

        if rebalance_mid_small_signs and n_small >= 4:
            total_sign = int(np.sum(small_signs))
            if abs(total_sign) > max(2, n_small // 2):
                target_sign = np.sign(total_sign)
                idx_perm = rng.permutation(n_small)
                flip_needed = abs(total_sign) // 2
                flipped = 0
                for j in idx_perm:
                    if np.sign(small_signs[j]) == target_sign:
                        small_signs[j] *= -1.0
                        flipped += 1
                        if flipped >= flip_needed:
                            break
    else:
        small_signs = np.array(
            [+1.0, +1.0, -1.0, -1.0] * ((n_small + 3) // 4),
            dtype=float
        )[:n_small]

    for k in range(n_small):
        xs, ys = _sample_pos(
            rng,
            Lx, Ly,
            margin=max(margin, 0.6 * small_core),
            existing_xy=existing_xy,
            min_dist=min_dist_small,
        )
        Gs = _rand_around(rng, Gamma_small_base, gamma_rel_std_small, 0.6, 1.4) * float(small_signs[k])
        Cs = _rand_around(rng, small_core, core_rel_std, 0.6, 1.4)
        vortices.append(Vortex(xs, ys, Gs, Cs))
        existing_xy.append((xs, ys))

    # --------------------------------------------------
    # 4) uniform flow
    # --------------------------------------------------
    if randomize_uniform:
        u_speed = float(rng.uniform(uniform_speed_min, uniform_speed_max))
        u_ang = float(rng.uniform(0.0, 360.0))
        th = np.deg2rad(u_ang)

        # angle convention: 0=+y, 90=+x
        uniform = (
            float(u_speed * np.sin(th)),
            float(u_speed * np.cos(th)),
        )
    else:
        uniform = default_uniform

    return vortices, uniform

def representative_center(vortices: List[Vortex], method: str = "absGamma") -> Tuple[float, float]:
    """
    method:
      - 'Gamma'    : weighted by Gamma (can cancel out if signs mix)
      - 'absGamma' : weighted by |Gamma| (stable)
    """
    if len(vortices) == 0:
        raise ValueError("No vortices provided.")
    w = np.array([v.Gamma for v in vortices], dtype=float)
    if method == "absGamma":
        w = np.abs(w)
    denom = np.sum(w)
    if denom == 0:
        # fallback: simple average
        xs = np.array([v.x0 for v in vortices])
        ys = np.array([v.y0 for v in vortices])
        return float(xs.mean()), float(ys.mean())

    xs = np.array([v.x0 for v in vortices])
    ys = np.array([v.y0 for v in vortices])
    return float(np.sum(w * xs) / denom), float(np.sum(w * ys) / denom)

@dataclass(slots=True)
class EdgeCache:
    N: int
    K: int
    E: int
    missing: int
    adj: np.ndarray        # (N,K) int32  neighbor or -1
    edge_id: np.ndarray    # (N,K) int32  directed edge id or -1
    dst_k_in: np.ndarray   # (E,)   int32  for edge e=(v->u), k_in at u such that adj[u,k_in]==v
    in_edge_id: np.ndarray # (N,K) int32  incoming edge id for state (v,k_in): (adj[v,k_in]
    src: np.ndarray        # (E,) int32
    dst: np.ndarray        # (E,) int32
    dx: np.ndarray         # (E,) float64
    dy: np.ndarray         # (E,) float64
    L: np.ndarray          # (E,) float64
    theta0: np.ndarray     # (E,) float64 rad

def build_edge_cache(vertices: np.ndarray, adj_fixed: np.ndarray, missing: int = -1) -> EdgeCache:
    """
    Build fixed-slot edge cache.

    Requirements:
      - adj_fixed is (N,K) with neighbor index or missing(-1)
      - adj_fixed should be symmetric (if u is neighbor of v, v should appear in u's slots)
        so that dst_k_in / in_edge_id can be constructed.

    Returns EdgeCacheFixed with:
      adj, edge_id, src, dst, dx, dy, L, theta0,
      dst_k_in, in_edge_id
    """
    vertices = np.asarray(vertices, dtype=np.float64)
    adj_fixed = np.asarray(adj_fixed, dtype=np.int32)

    if vertices.ndim != 2 or vertices.shape[1] != 2:
        raise ValueError("vertices must be (N,2).")
    N = vertices.shape[0]
    if adj_fixed.ndim != 2:
        raise ValueError("adj_fixed must be 2D (N,K).")
    if adj_fixed.shape[0] != N:
        raise ValueError("adj_fixed first dim must match vertices N.")
    K = int(adj_fixed.shape[1])

    valid = (adj_fixed != missing)
    E = int(valid.sum())

    src = np.empty(E, dtype=np.int32)
    dst = np.empty(E, dtype=np.int32)
    edge_id = np.full((N, K), missing, dtype=np.int32)

    # --- build directed edge list in (v, k) order ---
    e = 0
    for v in range(N):
        for k in range(K):
            u = int(adj_fixed[v, k])
            if u == missing:
                continue
            if not (0 <= u < N):
                raise ValueError(f"adj_fixed[{v},{k}]={u} out of range [0,{N}).")
            src[e] = v
            dst[e] = u
            edge_id[v, k] = e
            e += 1
    if e != E:
        raise RuntimeError("Edge count mismatch while building edge list.")

    # --- neighbor -> slot map (for fast reverse lookup) ---
    nbr2k: List[Dict[int, int]] = []
    for v in range(N):
        d: Dict[int, int] = {}
        for k in range(K):
            u = int(adj_fixed[v, k])
            if u != missing:
                d[u] = k
        nbr2k.append(d)

    # --- dst_k_in[e] : at destination u, which slot corresponds to coming from v? ---
    dst_k_in = np.full(E, missing, dtype=np.int32)
    for ee in range(E):
        v = int(src[ee])
        u = int(dst[ee])
        k_in = nbr2k[u].get(v, None)
        if k_in is None:
            raise RuntimeError(
                f"adj_fixed is not symmetric: node {u} has no slot for neighbor {v} "
                f"(needed for dst_k_in of edge {v}->{u})."
            )
        dst_k_in[ee] = int(k_in)

    # --- in_edge_id[v, k_in] : edge id of (adj_fixed[v,k_in] -> v) ---
    in_edge_id = np.full((N, K), missing, dtype=np.int32)
    for v in range(N):
        for k_in in range(K):
            pv = int(adj_fixed[v, k_in])
            if pv == missing:
                continue
            k_out = nbr2k[pv].get(v, None)
            if k_out is None:
                raise RuntimeError(
                    f"adj_fixed is not symmetric: node {pv} has no slot for neighbor {v} "
                    f"(needed for in_edge_id of state (v={v},k_in={k_in}))."
                )
            eid = int(edge_id[pv, int(k_out)])
            if eid == missing:
                raise RuntimeError("edge_id lookup failed unexpectedly (internal inconsistency).")
            in_edge_id[v, k_in] = eid

    # --- geometry ---
    dx = vertices[dst, 0] - vertices[src, 0]
    dy = vertices[dst, 1] - vertices[src, 1]
    L = np.hypot(dx, dy)
    if np.any(L == 0):
        raise ValueError("Zero-length edge found (duplicate vertex or self-loop).")
    theta0 = np.arctan2(dx, dy)  # rad, from +y clockwise

    return EdgeCache(
        N=N, K=K, E=E, missing=missing,
        adj=adj_fixed.astype(np.int32, copy=False),
        edge_id=edge_id,
        dst_k_in=dst_k_in,
        in_edge_id=in_edge_id,
        src=src, dst=dst,
        dx=dx, dy=dy, L=L, theta0=theta0
    )

def compute_theta_cost_from_cache(
    cache: EdgeCache,
    theta_f_deg: np.ndarray,   # (N,)
    v_f: np.ndarray,           # (N,)
    V: float,
    eps_denom: float = 1e-12,
    infeasible_to_nan: bool = True,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Vectorized per-edge compute using cached geometry.

    Returns:
      theta_deg: (E,) deg (from +y clockwise), infeasible -> nan (optional)
      denom:     (E,)
      cost:      (E,) invalid -> inf
      feasible:  (E,) bool
      valid:     (E,) bool (feasible & denom>eps_denom)
    """
    if V <= 0:
        raise ValueError("V must be positive.")
    if theta_f_deg.shape[0] != cache.N or v_f.shape[0] != cache.N:
        raise ValueError("theta_f_deg and v_f must have shape (N,) matching cache.N.")

    theta_f = np.deg2rad(theta_f_deg[cache.src])  # per-edge source current angle
    vf_e = v_f[cache.src]

    delta = theta_f - cache.theta0
    s = (vf_e / float(V)) * np.sin(delta)

    feasible = np.abs(s) <= 1.0
    s_clip = np.clip(s, -1.0, 1.0)

    # theta = theta0 - asin(s)
    theta = cache.theta0 - np.arcsin(s_clip)
    if infeasible_to_nan:
        theta = np.where(feasible, theta, np.nan)

    # denom = vf*cos(delta) + V*cos(theta0-theta)
    # cos(theta0-theta) = cos(asin(s)) = sqrt(1-s^2)
    cos_term = np.sqrt(np.maximum(0.0, 1.0 - s_clip**2))
    denom = vf_e * np.cos(delta) + float(V) * cos_term

    valid = feasible & (denom > eps_denom)

    cost = np.full(cache.E, np.inf, dtype=float)
    cost[valid] = cache.L[valid] / denom[valid]

    theta_deg = (np.rad2deg(theta) + 360.0) % 360.0
    return theta_deg, denom, cost, feasible, valid

def get_edge_id(cache: EdgeCache, v: int, k: int) -> int:
    return int(cache.edge_id[v, k])  # -1이면 missing

# ploting code
def plot_points_and_adjacency(
    vertices: np.ndarray,
    adjacency: List[List[int]] | None = None,
    rect: Tuple[float, float, float, float] | None = None,
    show_points: bool = True,
    show_indices: bool = False,
    draw_adjacency_edges: bool = False,
    point_size: float = 10.0,
    adj_line_width: float = 1.0,
    zoom: Tuple[float, float, float, float] | None = None,  # (x_min, x_max, y_min, y_max)
    pad: float = 0.0,
) -> None:
    x = vertices[:, 0]
    y = vertices[:, 1]
    fig, ax = plt.subplots()

    if draw_adjacency_edges and adjacency is not None:
        for v, neighs in enumerate(adjacency):
            for u in neighs:
                if u > v:
                    ax.plot([x[v], x[u]], [y[v], y[u]], linewidth=adj_line_width)

    if show_points:
        ax.scatter(x, y, s=point_size)

    if show_indices:
        for k, (xx, yy) in enumerate(vertices):
            ax.text(xx, yy, str(k), fontsize=8, ha="center", va="center")

    if rect is not None:
        x_s, x_f, y_s, y_f = rect
        ax.plot([x_s, x_f, x_f, x_s, x_s],
                [y_s, y_s, y_f, y_f, y_s],
                linewidth=1.2)

    # --- zoom ---
    if zoom is not None:
        x_min, x_max, y_min, y_max = zoom
        ax.set_xlim(x_min - pad, x_max + pad)
        ax.set_ylim(y_min - pad, y_max + pad)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Points + adjacency")
    plt.show()


def plot_current_quiver(vertices, ux, uy, stride=3, figsize=(6, 4), dpi=150,
                        point_size=10, show_points=False):
    x = vertices[::stride, 0]
    y = vertices[::stride, 1]
    u = ux[::stride]
    v = uy[::stride]

    plt.figure(figsize=figsize, dpi=dpi)

    # points
    if show_points:
        plt.scatter(x, y, s=point_size)

    # vectors
    plt.quiver(x, y, u, v)

    plt.gca().set_aspect("equal", adjustable="box")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Current field")
    plt.show()

def plot_adjacency_and_current(
    vertices: np.ndarray,               # (N,2)
    adjacency: List[List[int]],         # length N
    vortices: List[Vortex],
    uniform: Tuple[float, float] = (0.0, 0.0),
    rect: Tuple[float, float, float, float] | None = (0.0, 6.0, 0.0, 6.0),
    show_points: bool = True,
    show_indices: bool = False,
    draw_adjacency_edges: bool = True,
    quiver_stride: int = 1,
    quiver_scale: float | None = None,
    point_size: float = 12.0,
    adj_line_width: float = 0.9,
    vortex_marker_size: float = 120.0,
):
    """
    Draw:
      - vertices
      - adjacency edges (exactly as given)
      - current vectors computed from vortices (+ optional uniform)
      - vortex center markers
    """
    if vertices.ndim != 2 or vertices.shape[1] != 2:
        raise ValueError("vertices must be (N,2).")
    if len(adjacency) != vertices.shape[0]:
        raise ValueError("adjacency length must match number of vertices.")

    x = vertices[:, 0]
    y = vertices[:, 1]

    # current at vertices
    res = composite_current(vertices, vortices, uniform=uniform)
    ux,uy=res['total_ux'], res['total_uy']
    fig, ax = plt.subplots()

    # adjacency edges (use input as-is)
    if draw_adjacency_edges:
        for v, neighs in enumerate(adjacency):
            for u in neighs:
                if u > v:  # undirected draw once
                    ax.plot([x[v], x[u]], [y[v], y[u]], linewidth=adj_line_width)

    # points
    if show_points:
        ax.scatter(x, y, s=point_size)
    if show_indices:
        for k, (xx, yy) in enumerate(vertices):
            ax.text(xx, yy, str(k), fontsize=8, ha="center", va="center")

    # current vectors
    ax.quiver(
        x[::quiver_stride], y[::quiver_stride],
        ux[::quiver_stride], uy[::quiver_stride],
        scale=quiver_scale
    )



    # rectangle boundary
    if rect is not None:
        x_s, x_f, y_s, y_f = rect
        ax.plot([x_s, x_f, x_f, x_s, x_s],
                [y_s, y_s, y_f, y_f, y_s],
                linewidth=1.2)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Adjacency + current ")
    plt.show()



def plot_path_with_theta_and_current(
    vertices: np.ndarray,          # (N,2)
    cache,                         # EdgeCache
    node_path=None,                # list[int]
    edge_path=None,                # list[int]
    theta_deg_e=None,              # (E,) deg, from +y clockwise
    draw_all_edges: bool = False,
    show_nodes: bool = False,
    show_node_indices: bool = False,
    all_edges_stride: int = 1,
    # style (single unified color)
    edge_color="C0",
    all_edge_alpha: float = 0.20,
    all_edge_lw: float = 0.6,
    path_alpha: float = 0.95,
    path_lw: float = 2.5,
    # theta visualization (path heading)
    show_theta_quiver: bool = True,
    theta_on: str = "src",         # "mid" or "src"
    theta_arrow_len: float = 0.15,
    show_theta_text: bool = False,

    # --- current field (add this) ---
    show_current_quiver: bool = True,
    # Option A: provide components directly (N,)
    curr_ux: np.ndarray | None = None,
    curr_uy: np.ndarray | None = None,
    # Option B: provide magnitude+angle (N,) where angle is from +y clockwise
    curr_v: np.ndarray | None = None,
    curr_theta_deg: np.ndarray | None = None,

    current_stride: int = 1,            # 너무 빽빽하면 2,3...
    current_arrow_len: float = 0.20,    # data-unit length (조절)
    current_color: str = "#0033cc",     # 진한 푸른색
    current_alpha: float = 0.9,
    current_width: float = 0.003,

    figsize=(10, 8),
    dpi=150,
):
    if vertices.ndim != 2 or vertices.shape[1] != 2:
        raise ValueError("vertices must be (N,2).")

    x = vertices[:, 0]
    y = vertices[:, 1]

    plt.figure(figsize=figsize, dpi=dpi)

    # (0) current field (behind everything)
    if show_current_quiver:
        N = vertices.shape[0]
        stride = max(1, int(current_stride))
        idx = np.arange(0, N, stride)


        ux = np.asarray(curr_ux)[idx]
        uy = np.asarray(curr_uy)[idx]



        plt.quiver(
            x[idx], y[idx],
            ux, uy,
            color=current_color
        )

    # 1) 전체 간선(그래프)
    if draw_all_edges:
        E = cache.E
        step = max(1, int(all_edges_stride))
        for e in range(0, E, step):
            u = int(cache.src[e])
            v = int(cache.dst[e])
            plt.plot([x[u], x[v]], [y[u], y[v]],
                     color=edge_color, alpha=all_edge_alpha, linewidth=all_edge_lw, zorder=2)

    # 2) 노드 점
    if show_nodes:
        plt.scatter(x, y, s=10, alpha=0.8, color=edge_color, zorder=3)

    # 3) 경로
    if edge_path is not None and len(edge_path) > 0:
        for e in edge_path:
            u = int(cache.src[e])
            v = int(cache.dst[e])
            plt.plot([x[u], x[v]], [y[u], y[v]],
                     color=edge_color, alpha=path_alpha, linewidth=path_lw, zorder=4)

        s = int(cache.src[int(edge_path[0])])
        t = int(cache.dst[int(edge_path[-1])])
        plt.scatter([x[s]], [y[s]], s=80, marker="o", color=edge_color, zorder=5)
        plt.scatter([x[t]], [y[t]], s=80, marker="X", color=edge_color, zorder=5)

    elif node_path is not None and len(node_path) > 0:
        pts = np.array(node_path, dtype=int)
        plt.plot(x[pts], y[pts], color=edge_color, alpha=path_alpha, linewidth=path_lw, zorder=4)
        plt.scatter([x[pts[0]]], [y[pts[0]]], s=80, marker="o", color=edge_color, zorder=5)
        plt.scatter([x[pts[-1]]], [y[pts[-1]]], s=80, marker="X", color=edge_color, zorder=5)
    else:
        print("No path provided: pass node_path or edge_path.")

    # 4) path theta arrows (black)
    if show_theta_quiver and edge_path is not None and len(edge_path) > 0:
        if theta_deg_e is None:
            raise ValueError("theta_deg_e is required when show_theta_quiver=True.")

        pxs, pys, ux, uy = [], [], [], []
        for e in edge_path:
            th = float(theta_deg_e[e])
            if not np.isfinite(th):
                continue

            u0 = int(cache.src[e])
            v0 = int(cache.dst[e])

            if theta_on == "src":
                px, py = x[u0], y[u0]
            else:
                px, py = 0.5 * (x[u0] + x[v0]), 0.5 * (y[u0] + y[v0])

            th_rad = np.deg2rad(th)
            dir_x = np.sin(th_rad)
            dir_y = np.cos(th_rad)

            pxs.append(px); pys.append(py)
            ux.append(dir_x); uy.append(dir_y)

            if show_theta_text:
                plt.text(px, py, f"{th:.1f}°", fontsize=8, ha="left", va="bottom", color="black")

        if len(pxs) > 0:
            pxs = np.asarray(pxs); pys = np.asarray(pys)
            ux = np.asarray(ux); uy = np.asarray(uy)

            plt.quiver(
                pxs, pys,
                ux * theta_arrow_len, uy * theta_arrow_len,
                angles="xy", scale_units="xy", scale=1.0, width=0.003,
                color="black", alpha=path_alpha, zorder=6
            )

    # 5) 노드 인덱스
    if show_node_indices:
        for i in range(vertices.shape[0]):
            plt.text(x[i], y[i], str(i), fontsize=8, ha="center", va="center", color=edge_color, zorder=7)

    plt.gca().set_aspect("equal", adjustable="box")
    plt.grid(True, alpha=0.2)
    plt.show()


def plot_multi_path_with_theta_and_current(
    vertices: np.ndarray,          # (N,2)
    cache,                         # EdgeCache
    node_path=None,                # list[int] or list[list[int]]
    edge_path=None,                # list[int] or list[list[int]]
    theta_deg_e=None,              # (E,) deg, from +y clockwise
    draw_all_edges: bool = False,
    show_nodes: bool = False,
    show_node_indices: bool = False,
    all_edges_stride: int = 1,
    # style (single unified color)
    edge_color="C0",
    all_edge_alpha: float = 0.20,
    all_edge_lw: float = 0.6,
    path_alpha: float = 0.95,
    path_lw: float = 2.5,
    # theta visualization (path heading)
    show_theta_quiver: bool = True,
    theta_on: str = "src",         # "mid" or "src"
    theta_arrow_len: float = 0.15,
    show_theta_text: bool = False,

    # --- current field ---
    show_current_quiver: bool = True,
    curr_ux: np.ndarray | None = None,
    curr_uy: np.ndarray | None = None,
    curr_v: np.ndarray | None = None,
    curr_theta_deg: np.ndarray | None = None,

    current_stride: int = 1,
    current_arrow_len: float = 0.20,
    current_color: str = "#0033cc",
    current_alpha: float = 0.9,
    current_width: float = 0.003,

    # --- junction (connection) points ---
    show_junction_points: bool = True,
    junction_color: str = "red",
    junction_size: float = 70,
    junction_marker: str = "o",

    figsize=(10, 8),
    dpi=150,
):
    if vertices.ndim != 2 or vertices.shape[1] != 2:
        raise ValueError("vertices must be (N,2).")

    x = vertices[:, 0]
    y = vertices[:, 1]

    plt.figure(figsize=figsize, dpi=dpi)

    # (0) current field (behind everything)
    if show_current_quiver:
        N = vertices.shape[0]
        stride = max(1, int(current_stride))
        idx = np.arange(0, N, stride)


        ux = np.asarray(curr_ux)[idx]
        uy = np.asarray(curr_uy)[idx]



        plt.quiver(
            x[idx], y[idx],
            ux, uy,
            color=current_color
        )

    # 1) 전체 간선(그래프)
    if draw_all_edges:
        E = cache.E
        step = max(1, int(all_edges_stride))
        for e in range(0, E, step):
            u = int(cache.src[e])
            v = int(cache.dst[e])
            plt.plot([x[u], x[v]], [y[u], y[v]],
                     color=edge_color, alpha=all_edge_alpha, linewidth=all_edge_lw, zorder=2)

    # 2) 노드 점
    if show_nodes:
        plt.scatter(x, y, s=10, alpha=0.8, color=edge_color, zorder=3)

    # ---- helper: normalize multi paths ----
    def _is_list_of_lists(p):
        return isinstance(p, (list, tuple)) and len(p) > 0 and isinstance(p[0], (list, tuple, np.ndarray))

    edge_paths = None
    node_paths = None
    if edge_path is not None:
        edge_paths = edge_path if _is_list_of_lists(edge_path) else [edge_path]
        edge_paths = [list(p) for p in edge_paths if p is not None and len(p) > 0]
    if node_path is not None:
        node_paths = node_path if _is_list_of_lists(node_path) else [node_path]
        node_paths = [list(p) for p in node_paths if p is not None and len(p) > 0]

    # 3) 경로(멀티/단일 모두)
    junction_nodes = []

    if edge_paths is not None and len(edge_paths) > 0:
        # draw each segment
        for seg in edge_paths:
            for e in seg:
                u = int(cache.src[e])
                v = int(cache.dst[e])
                plt.plot([x[u], x[v]], [y[u], y[v]],
                         color=edge_color, alpha=path_alpha, linewidth=path_lw, zorder=4)

        # start / end
        s = int(cache.src[int(edge_paths[0][0])])
        t = int(cache.dst[int(edge_paths[-1][-1])])
        plt.scatter([x[s]], [y[s]], s=80, marker="o", color=edge_color, zorder=5)
        plt.scatter([x[t]], [y[t]], s=80, marker="X", color=edge_color, zorder=5)

        # junctions between segments (middle connection points)
        for i in range(len(edge_paths) - 1):
            prev_last = int(edge_paths[i][-1])
            next_first = int(edge_paths[i + 1][0])
            j1 = int(cache.dst[prev_last])      # end node of prev segment
            j2 = int(cache.src[next_first])     # start node of next segment
            junction_nodes.append(j1)
            junction_nodes.append(j2)

        # remove global endpoints from junction list
        junction_nodes = [j for j in junction_nodes if j not in (s, t)]
        # unique, keep order
        seen = set()
        junction_nodes = [j for j in junction_nodes if (j not in seen and not seen.add(j))]

        # plot junction points
        if show_junction_points and len(junction_nodes) > 0:
            plt.scatter(x[junction_nodes], y[junction_nodes],
                        s=junction_size, marker=junction_marker,
                        color=junction_color, edgecolors="none", zorder=6)

    elif node_paths is not None and len(node_paths) > 0:
        # draw each segment
        for seg in node_paths:
            pts = np.asarray(seg, dtype=int)
            plt.plot(x[pts], y[pts], color=edge_color, alpha=path_alpha, linewidth=path_lw, zorder=4)

        # start / end
        s = int(node_paths[0][0])
        t = int(node_paths[-1][-1])
        plt.scatter([x[s]], [y[s]], s=80, marker="o", color=edge_color, zorder=5)
        plt.scatter([x[t]], [y[t]], s=80, marker="X", color=edge_color, zorder=5)

        # junctions between segments
        for i in range(len(node_paths) - 1):
            j1 = int(node_paths[i][-1])
            j2 = int(node_paths[i + 1][0])
            junction_nodes.append(j1)
            junction_nodes.append(j2)

        junction_nodes = [j for j in junction_nodes if j not in (s, t)]
        seen = set()
        junction_nodes = [j for j in junction_nodes if (j not in seen and not seen.add(j))]

        if show_junction_points and len(junction_nodes) > 0:
            plt.scatter(x[junction_nodes], y[junction_nodes],
                        s=junction_size, marker=junction_marker,
                        color=junction_color, edgecolors="none", zorder=6)
    else:
        print("No path provided: pass node_path or edge_path (single or list-of-paths).")

    # 4) path theta arrows (black)  — edge_path 기반만 표시(멀티도 지원)
    if show_theta_quiver:
        if edge_paths is None or len(edge_paths) == 0:
            # node_path만 주어진 경우는 여기서 생략(원하면 node_path->edge_path 매핑 필요)
            pass
        else:
            if theta_deg_e is None:
                raise ValueError("theta_deg_e is required when show_theta_quiver=True.")

            pxs, pys, ux_list, uy_list = [], [], [], []
            for seg in edge_paths:
                for e in seg:
                    th = float(theta_deg_e[e])
                    if not np.isfinite(th):
                        continue

                    u0 = int(cache.src[e])
                    v0 = int(cache.dst[e])

                    if theta_on == "src":
                        px, py = x[u0], y[u0]
                    else:
                        px, py = 0.5 * (x[u0] + x[v0]), 0.5 * (y[u0] + y[v0])

                    th_rad = np.deg2rad(th)
                    dir_x = np.sin(th_rad)
                    dir_y = np.cos(th_rad)

                    pxs.append(px); pys.append(py)
                    ux_list.append(dir_x); uy_list.append(dir_y)

                    if show_theta_text:
                        plt.text(px, py, f"{th:.1f}°", fontsize=8, ha="left", va="bottom", color="black")

            if len(pxs) > 0:
                pxs = np.asarray(pxs); pys = np.asarray(pys)
                ux_list = np.asarray(ux_list); uy_list = np.asarray(uy_list)

                plt.quiver(
                    pxs, pys,
                    ux_list * theta_arrow_len, uy_list * theta_arrow_len,
                    angles="xy", scale_units="xy", scale=1.0, width=0.003,
                    color="black", alpha=path_alpha, zorder=7
                )

    # 5) 노드 인덱스
    if show_node_indices:
        for i in range(vertices.shape[0]):
            plt.text(x[i], y[i], str(i), fontsize=8, ha="center", va="center", color=edge_color, zorder=8)

    plt.gca().set_aspect("equal", adjustable="box")
    plt.grid(True, alpha=0.2)
    plt.show()

def plot_path(
    vertices: np.ndarray,                 # (N,2)
    node_path: List[int],                 # [v0,v1,...]
    *,
    show_all_points: bool = True,
    all_point_stride: int = 1,
    show_nodes: bool = True,
    show_edges: bool = True,
    show_arrows: bool = True,
    arrow_step: int = 1,
    annotate_nodes: bool = False,
    annotate_every: int = 1,
    start_marker: bool = True,
    goal_marker: bool = True,
    figsize: Tuple[float, float] = (7, 7),
    dpi: int = 150,
    pad_ratio: float = 0.25,
    title: Optional[str] = None,
):
    """
    Plot a node path on top of vertices.

    - 줌은 path 주변으로 자동 설정 (pad_ratio 만큼 여유)
    - node_path가 비어있으면 빈 플롯만 보여줌
    """
    if vertices.ndim != 2 or vertices.shape[1] != 2:
        raise ValueError("vertices must be (N,2).")

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    # background points
    if show_all_points:
        stride = max(1, int(all_point_stride))
        pts = vertices[::stride]
        ax.scatter(pts[:, 0], pts[:, 1], s=8)

    if node_path is None or len(node_path) == 0:
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_title(title or "Empty path")
        plt.show()
        return fig, ax

    path = np.asarray(node_path, dtype=np.int64)
    P = vertices[path]  # (L,2)

    # path edges
    if show_edges and len(path) >= 2:
        ax.plot(P[:, 0], P[:, 1], linewidth=2.0)

    # path nodes
    if show_nodes:
        ax.scatter(P[:, 0], P[:, 1], s=40)

    # arrows along the path
    if show_arrows and len(path) >= 2:
        step = max(1, int(arrow_step))
        for i in range(0, len(path) - 1, step):
            x0, y0 = P[i]
            x1, y1 = P[i + 1]
            dx, dy = (x1 - x0), (y1 - y0)
            ax.arrow(
                float(x0), float(y0), float(dx), float(dy),
                length_includes_head=True, head_width=0.0, head_length=0.0
            )

    # annotate nodes
    if annotate_nodes:
        every = max(1, int(annotate_every))
        for i in range(0, len(path), every):
            v = int(path[i])
            x, y = P[i]
            ax.text(float(x), float(y), f"{v}", fontsize=9)

    # start/goal markers
    if start_marker:
        ax.scatter([P[0, 0]], [P[0, 1]], s=120, marker="o")
    if goal_marker:
        ax.scatter([P[-1, 0]], [P[-1, 1]], s=140, marker="X")

    # zoom around path
    xmin, xmax = float(P[:, 0].min()), float(P[:, 0].max())
    ymin, ymax = float(P[:, 1].min()), float(P[:, 1].max())
    dx = max(xmax - xmin, 1e-9)
    dy = max(ymax - ymin, 1e-9)
    ax.set_xlim(xmin - pad_ratio * dx, xmax + pad_ratio * dx)
    ax.set_ylim(ymin - pad_ratio * dy, ymax + pad_ratio * dy)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title or f"Path (len={len(path)})")

    plt.show()

import numpy as np
import heapq
from typing import List, Tuple

def wrap180_deg(a: float) -> float:
    return (float(a) + 180.0) % 360.0 - 180.0

def ang_diff_deg(a_deg: float, b_deg: float) -> float:
    return wrap180_deg(float(a_deg) - float(b_deg))

def turn_penalty_deg(delta_deg: float,
                     th1: float, lam1: float,
                     th2: float, lam2: float) -> float:
    """
    2-stage proportional penalty (piecewise linear):
      - |delta| <= th1 : 0
      - th1 < |delta| <= th2 : lam1 * (|delta|-th1)
      - |delta| > th2 : lam1*(th2-th1) + lam2*(|delta|-th2)

    여기서 lam1, lam2는 'deg당 비용' (cost per degree) 의미.
    """
    ad = abs(float(delta_deg))
    if ad <= th1:
        return 0.0
    if ad <= th2:
        return float(lam1) * (ad - th1)
    return float(lam2)

def dijkstra_turn_state_core(
    cache: EdgeCache,
    base_cost_e: np.ndarray,     # (E,)
    theta_e_deg: np.ndarray,     # (E,) (+y clockwise, deg)
    start: int,
    goal: int,

    th1: float = 40.0, lam1: float = 10,
    th2: float = 50.0, lam2: float = np.inf,

    use_start_heading: bool = False,
    start_heading_deg: float = 0.0,

    termination: Literal["goal_best", "goal_allk", "all"] = "goal_best",
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float, int]:
    """
    Fixed(K) Dijkstra over state (v, k_in).

    Uses:
      cache.adj (N,K), cache.edge_id (N,K),
      cache.dst_k_in (E,), cache.in_edge_id (N,K)
    """
    N = int(cache.N)
    K = int(cache.K)
    missing = int(cache.missing)

    if base_cost_e.shape[0] != cache.E or theta_e_deg.shape[0] != cache.E:
        raise ValueError("base_cost_e/theta_e_deg must have shape (E,) matching cache.E.")

    valid_in = (cache.adj != missing)  # (N,K)

    INF = float("inf")
    dist = np.full((N, K), INF, dtype=float)
    prev_node = np.full((N, K), -1, dtype=np.int32)
    prev_k    = np.full((N, K), -1, dtype=np.int32)

    if start == goal and termination !="all":
        return dist, prev_node, prev_k

    # goal_allk: 유효한 k만 세야 함
    goal_valid = valid_in[goal]
    goal_settled = np.zeros(K, dtype=bool)
    goal_settled_cnt = 0
    goal_valid_cnt = int(goal_valid.sum())

    h: list[tuple[float, int, int]] = []  # (d, v, k_in)

    # --- init from start (start -> u) ---
    for k_out in range(K):
        e = int(cache.edge_id[start, k_out])
        if e == missing:
            continue
        u = int(cache.adj[start, k_out])
        if u == missing:
            continue

        k_in_u = int(cache.dst_k_in[e])
        if not (0 <= k_in_u < K) or not bool(valid_in[u, k_in_u]):
            continue

        c = float(base_cost_e[e])
        th_out = float(theta_e_deg[e])
        if not np.isfinite(c) or not np.isfinite(th_out):
            continue

        cost = c
        if use_start_heading:
            delta0 = wrap180_deg(th_out - float(start_heading_deg))
            cost += turn_penalty_deg(delta0, th1, lam1, th2, lam2)

        if cost < dist[u, k_in_u]:
            dist[u, k_in_u] = cost
            prev_node[u, k_in_u] = int(start)
            prev_k[u, k_in_u] = -1
            heapq.heappush(h, (cost, u, k_in_u))

    best_goal = INF
    best_goal_k = -1

    pop_cnt = 0
    while h:
        d, v, k_in = heapq.heappop(h)
        if d != dist[v, k_in]:
            continue
        pop_cnt += 1

        if v == goal:
            if d < best_goal:
                best_goal = float(d)
                best_goal_k = int(k_in)

            if goal_valid[k_in] and not goal_settled[k_in]:
                goal_settled[k_in] = True
                goal_settled_cnt += 1

            if termination == "goal_best":
                #print("STOP at goal_best, pop_cnt =", pop_cnt)
                break
            if termination == "goal_allk" and goal_settled_cnt == goal_valid_cnt:
                break
            continue

        # incoming edge (prev -> v) for this state
        e_in = int(cache.in_edge_id[v, k_in])
        if e_in == missing:
            continue
        theta_in = float(theta_e_deg[e_in])
        if not np.isfinite(theta_in):
            continue

        # outgoing by fixed slots
        for k_out in range(K):
            e_out = int(cache.edge_id[v, k_out])
            if e_out == missing:
                continue
            u = int(cache.adj[v, k_out])
            if u == missing:
                continue

            k_in_u = int(cache.dst_k_in[e_out])
            if not (0 <= k_in_u < K) or not bool(valid_in[u, k_in_u]):
                continue

            c = float(base_cost_e[e_out])
            th_out = float(theta_e_deg[e_out])
            if not np.isfinite(c) or not np.isfinite(th_out):
                continue

            delta = wrap180_deg(th_out - theta_in)
            nd = float(d) + c + turn_penalty_deg(delta, th1, lam1, th2, lam2)

            if nd < dist[u, k_in_u]:
                dist[u, k_in_u] = nd
                prev_node[u, k_in_u] = int(v)
                prev_k[u, k_in_u] = int(k_in)
                heapq.heappush(h, (nd, u, k_in_u))

    return dist, prev_node, prev_k, float(best_goal), int(best_goal_k)

def reconstruct_path_from_prev(
    cache,
    prev_node: np.ndarray,   # (N,K) int32
    prev_k: np.ndarray,      # (N,K) int32
    start: int,
    goal: int,
    best_goal_k: int,
    *,
    use_k: bool = False,
    fixed_k: int = 0,
) -> Tuple[List[int], List[int], List[int]]:
    """
    Reconstruct (node_path, edge_path, k_in_path) from FIXED (N,K) prev arrays.

    Assumed cache fields (fixed version):
      - N: int
      - K: int
      - missing: int (typically -1)
      - in_edge_id: (N,K) int32 : incoming edge id for state (v,k_in) meaning (adj[v,k_in] -> v), or missing
    """
    if start == goal:
        return [start], [], []

    N = int(cache.N)
    K = int(cache.K)
    missing = int(getattr(cache, "missing", -1))

    if best_goal_k < 0:
        return [], [], []

    v = int(goal)
    k = int(fixed_k) if use_k else int(best_goal_k)
    if not (0 <= k < K):
        return [], [], []

    node_path_rev: List[int] = []
    k_path_rev: List[int] = []
    edge_path_rev: List[int] = []

    while True:
        node_path_rev.append(v)
        k_path_rev.append(k)

        pv = int(prev_node[v, k])
        pk = int(prev_k[v, k])

        # Collect incoming edge id for current state (pv -> v)
        e_in = int(cache.in_edge_id[v, k]) if hasattr(cache, "in_edge_id") else missing
        if e_in != missing:
            edge_path_rev.append(e_in)

        if pv < 0:
            break

        # reached the first hop from start
        if pv == start and pk == -1:
            node_path_rev.append(int(start))
            break

        v, k = pv, pk
        if not (0 <= k < K):
            # corrupted prev pointers
            return [], [], []

    node_path = node_path_rev[::-1]

    # k_path: start 다음 노드부터의 k_in (각 노드에 "어디서 들어왔는지" 슬롯)
    k_path = k_path_rev[::-1]
    if len(k_path) == len(node_path):
        k_path = k_path[1:]

    # edge_path: reverse로 쌓인 (prev->cur) 들을 뒤집기
    edge_path = edge_path_rev[::-1]

    # sanity: edge_path length should match node_path-1 (allow mismatch if cache.in_edge_id missing)
    if edge_path and len(edge_path) != len(node_path) - 1:
        # if mismatch, return without edges rather than returning wrong ones
        edge_path = []

    return node_path, edge_path, k_path

import numpy as np

import numpy as np
from typing import Optional, Tuple

def last_heading_from_edge_path(theta_e_deg: np.ndarray, edge_path: list[int]) -> float:
    if not edge_path:
        raise ValueError("edge_path is empty; cannot get last heading.")
    return float(theta_e_deg[int(edge_path[-1])])

def sample_random_points(
    n: int,
    start: Tuple[float, float] = (300.0, 300.0),
    goal: Tuple[float, float]  = (3300.0, 1500.0),
    x_range: Tuple[float, float] = (0.0, 3600.0),
    y_range: Tuple[float, float] = (0.0, 1800.0),
    min_dist: float = 0.0,
    seed: Optional[int] = 42,
    *,
    dtype=np.float64,
    max_tries: int = 300000,
    batch: int = 1024,
) -> np.ndarray:
    """
    Returns:
        points: (n,2) array
          - points[0]  = start
          - points[-1] = goal
          - all pairwise distances >= min_dist (if min_dist > 0)
    """
    if n < 0:
        raise ValueError("n must be >= 0")
    if min_dist < 0:
        raise ValueError("min_dist must be >= 0")

    x0, x1 = x_range
    y0, y1 = y_range
    if x1 < x0:
        x0, x1 = x1, x0
    if y1 < y0:
        y0, y1 = y1, y0

    rng = np.random.default_rng(seed)

    if n == 0:
        return np.empty((0, 2), dtype=dtype)
    if n == 1:
        # 정책: 1개면 start만
        return np.array([[float(start[0]), float(start[1])]], dtype=dtype)

    sx, sy = float(start[0]), float(start[1])
    gx, gy = float(goal[0]), float(goal[1])

    # start/goal 범위 체크
    if not (x0 <= sx <= x1 and y0 <= sy <= y1):
        raise ValueError(f"start={start} is outside x_range/y_range.")
    if not (x0 <= gx <= x1 and y0 <= gy <= y1):
        raise ValueError(f"goal={goal} is outside x_range/y_range.")

    # min_dist면 start-goal 거리도 만족해야 함
    if min_dist > 0.0:
        d2 = (sx - gx) ** 2 + (sy - gy) ** 2
        if d2 < min_dist ** 2:
            raise ValueError(
                f"start and goal are closer than min_dist. dist={d2**0.5:.3f}, min_dist={min_dist}"
            )

    # 결과 배열: start, (random...), goal
    points = np.empty((n, 2), dtype=np.float64)
    points[0] = (sx, sy)
    points[-1] = (gx, gy)

    if n == 2:
        return points.astype(dtype, copy=False)

    # min_dist 없으면 가운데만 그냥 샘플링
    if min_dist <= 0.0:
        xs = rng.uniform(x0, x1, size=n - 2)
        ys = rng.uniform(y0, y1, size=n - 2)
        points[1:-1, 0] = xs
        points[1:-1, 1] = ys
        return points.astype(dtype, copy=False)

    min2 = float(min_dist * min_dist)

    # 채워진 점들의 리스트(거리 체크용): start + goal은 이미 들어있다고 보고 시작
    # points_filled에는 현재까지 확정된 점들을 모아두고, 마지막에 points[1:-1]에 채운다.
    filled = np.empty((n, 2), dtype=np.float64)
    filled[0] = (sx, sy)
    filled[1] = (gx, gy)
    k = 2  # filled에 들어있는 개수 (start, goal)

    out_mid = np.empty((n - 2, 2), dtype=np.float64)
    mcount = 0  # mid points 개수

    tries = 0
    while mcount < (n - 2) and tries < max_tries:
        m = min(batch, (n - 2) - mcount)
        cand_x = rng.uniform(x0, x1, size=m)
        cand_y = rng.uniform(y0, y1, size=m)
        cand = np.column_stack([cand_x, cand_y])

        for i in range(m):
            # filled(= start, goal, 그리고 이미 채택된 mid들)과의 거리 체크
            dx = filled[:k, 0] - cand[i, 0]
            dy = filled[:k, 1] - cand[i, 1]
            if np.min(dx * dx + dy * dy) >= min2:
                out_mid[mcount] = cand[i]
                filled[k] = cand[i]
                k += 1
                mcount += 1
                if mcount >= (n - 2):
                    break

        tries += m

    if mcount < (n - 2):
        raise RuntimeError(
            f"Failed to sample {n} points with min_dist={min_dist} "
            f"within max_tries={max_tries}. Only sampled {mcount + 2} points total. "
            f"(Try lowering min_dist or n, or increasing max_tries.)"
        )

    points[1:-1] = out_mid
    return points.astype(dtype, copy=False)

import heapq
import time
from typing import Callable, Dict, Any, Optional


# =========================================================
# A* heuristic helpers
# =========================================================
def euclidean_time_heuristic(
    *,
    points: np.ndarray,
    node_idx: int,
    goal_idx: int,
    usv_speed: float,
) -> float:
    """
    h(v) = ||x_goal - x_v|| / speed

    주의:
    해류가 순방향으로 강하면 실제 최적 시간이 이보다 더 작아질 수 있어서
    admissible 보장이 깨질 수 있다.
    즉, '빠른 근사' 비교용으로는 좋지만, 최적성 보장은 약하다.
    """
    if usv_speed <= 0:
        raise ValueError("usv_speed must be positive.")
    dist = float(np.linalg.norm(points[goal_idx] - points[node_idx]))
    return dist / usv_speed


def predict_surrogate_travel_time(
    *,
    model,
    feature_names: List[str],
    target_mode: str,
    points: np.ndarray,
    blocked_mask: np.ndarray,
    total_ux: np.ndarray,
    total_uy: np.ndarray,
    start_idx: int,
    goal_idx: int,
    start_heading_deg: float,
    usv_speed: float,
    inner_width: float = 90.0,
    outer_width: float = 180.0,
    clip_nonnegative: bool = True,
) -> float:
    """
    학습된 surrogate 모델로 start -> goal travel time 예측
    """
    feat = extract_fixed_features_with_outer_gap(
        points=points,
        blocked_mask=blocked_mask,
        total_ux=total_ux,
        total_uy=total_uy,
        start_idx=start_idx,
        goal_idx=goal_idx,
        start_heading_deg=start_heading_deg,
        inner_width=inner_width,
        outer_width=outer_width,
    )

    x = np.array([[feat[name] for name in feature_names]], dtype=float)
    y_pred = np.asarray(model.predict(x), dtype=float).reshape(-1)

    base_dist = float(np.linalg.norm(points[goal_idx] - points[start_idx]))
    y_base = np.array([base_dist / usv_speed], dtype=float)

    pred_time = float(recover_time_from_target(y_pred, y_base, target_mode)[0])

    if clip_nonnegative:
        pred_time = max(pred_time, 0.0)

    return pred_time


# =========================================================
# generic A* over turn-state (v, k_in)
# =========================================================
def astar_turn_state_core(
    cache: EdgeCache,
    base_cost_e: np.ndarray,     # (E,)
    theta_e_deg: np.ndarray,     # (E,)
    start: int,
    goal: int,
    heuristic_fn: Callable[[int, int], float],  # h(v, k_in) -> estimated remaining time
    *,
    th1: float = 40.0,
    lam1: float = 10.0,
    th2: float = 50.0,
    lam2: float = np.inf,
    use_start_heading: bool = False,
    start_heading_deg: float = 0.0,
    termination: Literal["goal_best", "goal_allk", "all"] = "goal_best",
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float, int]:
    """
    Dijkstra와 동일한 상태공간 (v, k_in)에서 동작하는 A*
    priority = g + h
    """
    N = int(cache.N)
    K = int(cache.K)
    missing = int(cache.missing)

    if base_cost_e.shape[0] != cache.E or theta_e_deg.shape[0] != cache.E:
        raise ValueError("base_cost_e/theta_e_deg must have shape (E,) matching cache.E.")

    valid_in = (cache.adj != missing)

    INF = float("inf")
    dist = np.full((N, K), INF, dtype=float)       # g-cost
    prev_node = np.full((N, K), -1, dtype=np.int32)
    prev_k = np.full((N, K), -1, dtype=np.int32)

    if start == goal and termination != "all":
        return dist, prev_node, prev_k, 0.0, -1

    goal_valid = valid_in[goal]
    goal_settled = np.zeros(K, dtype=bool)
    goal_settled_cnt = 0
    goal_valid_cnt = int(goal_valid.sum())

    # heap item: (f, g, v, k_in)
    hq: list[tuple[float, float, int, int]] = []

    # --- init: start -> u ---
    for k_out in range(K):
        e = int(cache.edge_id[start, k_out])
        if e == missing:
            continue

        u = int(cache.adj[start, k_out])
        if u == missing:
            continue

        k_in_u = int(cache.dst_k_in[e])
        if not (0 <= k_in_u < K) or not bool(valid_in[u, k_in_u]):
            continue

        c = float(base_cost_e[e])
        th_out = float(theta_e_deg[e])
        if not np.isfinite(c) or not np.isfinite(th_out):
            continue

        g_new = c
        if use_start_heading:
            delta0 = wrap180_deg(th_out - float(start_heading_deg))
            g_new += turn_penalty_deg(delta0, th1, lam1, th2, lam2)

        if g_new < dist[u, k_in_u]:
            dist[u, k_in_u] = g_new
            prev_node[u, k_in_u] = int(start)
            prev_k[u, k_in_u] = -1

            h_val = float(heuristic_fn(u, k_in_u))
            if not np.isfinite(h_val):
                h_val = 0.0
            heapq.heappush(hq, (g_new + h_val, g_new, u, k_in_u))

    best_goal = INF
    best_goal_k = -1

    while hq:
        f, g, v, k_in = heapq.heappop(hq)

        if g != dist[v, k_in]:
            continue

        if v == goal:
            if g < best_goal:
                best_goal = float(g)
                best_goal_k = int(k_in)

            if goal_valid[k_in] and not goal_settled[k_in]:
                goal_settled[k_in] = True
                goal_settled_cnt += 1

            if termination == "goal_best":
                break
            if termination == "goal_allk" and goal_settled_cnt == goal_valid_cnt:
                break
            continue

        e_in = int(cache.in_edge_id[v, k_in])
        if e_in == missing:
            continue

        theta_in = float(theta_e_deg[e_in])
        if not np.isfinite(theta_in):
            continue

        for k_out in range(K):
            e_out = int(cache.edge_id[v, k_out])
            if e_out == missing:
                continue

            u = int(cache.adj[v, k_out])
            if u == missing:
                continue

            k_in_u = int(cache.dst_k_in[e_out])
            if not (0 <= k_in_u < K) or not bool(valid_in[u, k_in_u]):
                continue

            c = float(base_cost_e[e_out])
            th_out = float(theta_e_deg[e_out])
            if not np.isfinite(c) or not np.isfinite(th_out):
                continue

            delta = wrap180_deg(th_out - theta_in)
            g_new = float(g) + c + turn_penalty_deg(delta, th1, lam1, th2, lam2)

            if g_new < dist[u, k_in_u]:
                dist[u, k_in_u] = g_new
                prev_node[u, k_in_u] = int(v)
                prev_k[u, k_in_u] = int(k_in)

                h_val = float(heuristic_fn(u, k_in_u))
                if not np.isfinite(h_val):
                    h_val = 0.0

                heapq.heappush(hq, (g_new + h_val, g_new, u, k_in_u))

    return dist, prev_node, prev_k, float(best_goal), int(best_goal_k)


# =========================================================
# wrapper 1: A* with Euclidean/speed heuristic
# =========================================================
def shortest_path_between_vertices_astar_euclidean(
    *,
    points: np.ndarray,
    cache: EdgeCache,
    cost_e: np.ndarray,
    theta_e_deg: np.ndarray,
    start_vid: int,
    goal_vid: int,
    usv_speed: float,
    th1: float = 40.0,
    lam1: float = 10.0,
    th2: float = 50.0,
    lam2: float = np.inf,
    use_start_heading: bool = False,
    start_heading_deg: float = 0.0,
    termination: Literal["goal_best", "goal_allk", "all"] = "goal_best",
) -> Dict[str, Any]:

    def heuristic_fn(v: int, k_in: int) -> float:
        return euclidean_time_heuristic(
            points=points,
            node_idx=v,
            goal_idx=goal_vid,
            usv_speed=usv_speed,
        )

    dist, prev_node, prev_k, best_goal, best_goal_k = astar_turn_state_core(
        cache=cache,
        base_cost_e=cost_e,
        theta_e_deg=theta_e_deg,
        start=start_vid,
        goal=goal_vid,
        heuristic_fn=heuristic_fn,
        th1=th1, lam1=lam1,
        th2=th2, lam2=lam2,
        use_start_heading=use_start_heading,
        start_heading_deg=start_heading_deg,
        termination=termination,
    )

    node_path, edge_path, k_path = reconstruct_path_from_prev(
        cache=cache,
        prev_node=prev_node,
        prev_k=prev_k,
        start=start_vid,
        goal=goal_vid,
        best_goal_k=best_goal_k,
    )

    return {
        "dist": dist,
        "prev_node": prev_node,
        "prev_k": prev_k,
        "best_goal": best_goal,
        "best_goal_k": best_goal_k,
        "node_path": node_path,
        "edge_path": edge_path,
        "k_path": k_path,
    }


# =========================================================
# wrapper 2: A* with learned surrogate heuristic
# =========================================================
def shortest_path_between_vertices_astar_surrogate(
    *,
    points: np.ndarray,
    blocked_mask: np.ndarray,
    total_ux: np.ndarray,
    total_uy: np.ndarray,
    cache: EdgeCache,
    cost_e: np.ndarray,
    theta_e_deg: np.ndarray,
    start_vid: int,
    goal_vid: int,
    surrogate_model,
    feature_names: List[str],
    target_mode: str,
    usv_speed: float,
    inner_width: float = 90.0,
    outer_width: float = 180.0,
    th1: float = 40.0,
    lam1: float = 10.0,
    th2: float = 50.0,
    lam2: float = np.inf,
    use_start_heading: bool = False,
    start_heading_deg: float = 0.0,
    termination: Literal["goal_best", "goal_allk", "all"] = "goal_best",
    heuristic_clip_nonnegative: bool = True,
) -> Dict[str, Any]:

    heuristic_cache: Dict[Tuple[int, int], float] = {}

    def state_heading_deg(v: int, k_in: int) -> float:
        e_in = int(cache.in_edge_id[v, k_in])
        if e_in >= 0 and np.isfinite(theta_e_deg[e_in]):
            return float(theta_e_deg[e_in])
        return float(start_heading_deg)

    def heuristic_fn(v: int, k_in: int) -> float:
        key = (int(v), int(k_in))
        if key in heuristic_cache:
            return heuristic_cache[key]

        h_val = predict_surrogate_travel_time(
            model=surrogate_model,
            feature_names=feature_names,
            target_mode=target_mode,
            points=points,
            blocked_mask=blocked_mask,
            total_ux=total_ux,
            total_uy=total_uy,
            start_idx=v,
            goal_idx=goal_vid,
            start_heading_deg=state_heading_deg(v, k_in),
            usv_speed=usv_speed,
            inner_width=inner_width,
            outer_width=outer_width,
            clip_nonnegative=heuristic_clip_nonnegative,
        )
        heuristic_cache[key] = float(h_val)
        return float(h_val)

    dist, prev_node, prev_k, best_goal, best_goal_k = astar_turn_state_core(
        cache=cache,
        base_cost_e=cost_e,
        theta_e_deg=theta_e_deg,
        start=start_vid,
        goal=goal_vid,
        heuristic_fn=heuristic_fn,
        th1=th1, lam1=lam1,
        th2=th2, lam2=lam2,
        use_start_heading=use_start_heading,
        start_heading_deg=start_heading_deg,
        termination=termination,
    )

    node_path, edge_path, k_path = reconstruct_path_from_prev(
        cache=cache,
        prev_node=prev_node,
        prev_k=prev_k,
        start=start_vid,
        goal=goal_vid,
        best_goal_k=best_goal_k,
    )

    return {
        "dist": dist,
        "prev_node": prev_node,
        "prev_k": prev_k,
        "best_goal": best_goal,
        "best_goal_k": best_goal_k,
        "node_path": node_path,
        "edge_path": edge_path,
        "k_path": k_path,
        "heuristic_cache_size": len(heuristic_cache),
    }


# =========================================================
# optional benchmark helper for 4-way comparison
# =========================================================
def compare_four_methods_one_pair(
    *,
    points: np.ndarray,
    blocked_mask: np.ndarray,
    total_ux: np.ndarray,
    total_uy: np.ndarray,
    cache: EdgeCache,
    cost_e: np.ndarray,
    theta_e_deg: np.ndarray,
    start_vid: int,
    goal_vid: int,
    usv_speed: float,
    learned_model=None,          # DT / Ridge / RF 등
    learned_feature_names=None,
    learned_target_mode: str = "absolute",
    surrogate_model=None,        # DT-based surrogate for A*
    surrogate_feature_names=None,
    surrogate_target_mode: str = "absolute",
    inner_width: float = 90.0,
    outer_width: float = 180.0,
    th1: float = 40.0,
    lam1: float = 10.0,
    th2: float = 50.0,
    lam2: float = np.inf,
    use_start_heading: bool = False,
    start_heading_deg: float = 0.0,
) -> Dict[str, Dict[str, Any]]:
    """
    1. Dijkstra baseline
    2. learned model only (path search 아님, travel time prediction only)
    3. A* with euclidean/speed heuristic
    4. A* with learned surrogate heuristic
    """

    out = {}

    # -------------------------------------------------
    # 1) Dijkstra baseline
    # -------------------------------------------------
    t0 = time.perf_counter()
    res_dij = shortest_path_between_vertices(
        cache=cache,
        cost_e=cost_e,
        theta_e_deg=theta_e_deg,
        start_vid=start_vid,
        goal_vid=goal_vid,
        th1=th1, lam1=lam1,
        th2=th2, lam2=lam2,
        use_start_heading=use_start_heading,
        start_heading_deg=start_heading_deg,
        termination="goal_best",
    )
    t1 = time.perf_counter()

    out["dijkstra"] = {
        "travel_time": float(res_dij["best_goal"]),
        "runtime_sec": float(t1 - t0),
        "node_path": res_dij["node_path"],
        "edge_path": res_dij["edge_path"],
    }

    # -------------------------------------------------
    # 2) learned model only
    # -------------------------------------------------
    if learned_model is not None:
        t0 = time.perf_counter()
        pred_time = predict_surrogate_travel_time(
            model=learned_model,
            feature_names=learned_feature_names,
            target_mode=learned_target_mode,
            points=points,
            blocked_mask=blocked_mask,
            total_ux=total_ux,
            total_uy=total_uy,
            start_idx=start_vid,
            goal_idx=goal_vid,
            start_heading_deg=start_heading_deg,
            usv_speed=usv_speed,
            inner_width=inner_width,
            outer_width=outer_width,
        )
        t1 = time.perf_counter()

        out["learned_direct_prediction"] = {
            "predicted_travel_time": float(pred_time),
            "runtime_sec": float(t1 - t0),
            "abs_error_vs_dijkstra": float(abs(pred_time - res_dij["best_goal"])),
            "rel_error_vs_dijkstra_percent": float(
                100.0 * abs(pred_time - res_dij["best_goal"]) / max(abs(res_dij["best_goal"]), 1e-12)
            ),
        }

    # -------------------------------------------------
    # 3) A* with Euclidean / speed heuristic
    # -------------------------------------------------
    t0 = time.perf_counter()
    res_astar_euc = shortest_path_between_vertices_astar_euclidean(
        points=points,
        cache=cache,
        cost_e=cost_e,
        theta_e_deg=theta_e_deg,
        start_vid=start_vid,
        goal_vid=goal_vid,
        usv_speed=usv_speed,
        th1=th1, lam1=lam1,
        th2=th2, lam2=lam2,
        use_start_heading=use_start_heading,
        start_heading_deg=start_heading_deg,
        termination="goal_best",
    )
    t1 = time.perf_counter()

    out["astar_euclidean"] = {
        "travel_time": float(res_astar_euc["best_goal"]),
        "runtime_sec": float(t1 - t0),
        "path_gap_vs_dijkstra": float(res_astar_euc["best_goal"] - res_dij["best_goal"]),
        "rel_gap_vs_dijkstra_percent": float(
            100.0 * (res_astar_euc["best_goal"] - res_dij["best_goal"]) / max(abs(res_dij["best_goal"]), 1e-12)
        ),
        "node_path": res_astar_euc["node_path"],
        "edge_path": res_astar_euc["edge_path"],
    }

    # -------------------------------------------------
    # 4) A* with learned surrogate heuristic
    # -------------------------------------------------
    if surrogate_model is not None:
        t0 = time.perf_counter()
        res_astar_sur = shortest_path_between_vertices_astar_surrogate(
            points=points,
            blocked_mask=blocked_mask,
            total_ux=total_ux,
            total_uy=total_uy,
            cache=cache,
            cost_e=cost_e,
            theta_e_deg=theta_e_deg,
            start_vid=start_vid,
            goal_vid=goal_vid,
            surrogate_model=surrogate_model,
            feature_names=surrogate_feature_names,
            target_mode=surrogate_target_mode,
            usv_speed=usv_speed,
            inner_width=inner_width,
            outer_width=outer_width,
            th1=th1, lam1=lam1,
            th2=th2, lam2=lam2,
            use_start_heading=use_start_heading,
            start_heading_deg=start_heading_deg,
            termination="goal_best",
        )
        t1 = time.perf_counter()

        out["astar_surrogate"] = {
            "travel_time": float(res_astar_sur["best_goal"]),
            "runtime_sec": float(t1 - t0),
            "path_gap_vs_dijkstra": float(res_astar_sur["best_goal"] - res_dij["best_goal"]),
            "rel_gap_vs_dijkstra_percent": float(
                100.0 * (res_astar_sur["best_goal"] - res_dij["best_goal"]) / max(abs(res_dij["best_goal"]), 1e-12)
            ),
            "heuristic_cache_size": int(res_astar_sur["heuristic_cache_size"]),
            "node_path": res_astar_sur["node_path"],
            "edge_path": res_astar_sur["edge_path"],
        }

    return out

def build_vertex_radius_index(vertices: np.ndarray):
    """
    Fast radius queries using KDTree (recommended).
    Returns a cKDTree if SciPy is available, else None.
    """
    if cKDTree is None:
        return None
    vertices = np.asarray(vertices, dtype=np.float64)
    if vertices.ndim != 2 or vertices.shape[1] != 2:
        raise ValueError("vertices must be (N,2).")
    return cKDTree(vertices)


def neighbors_within_radius(
    vertices: np.ndarray,          # (N,2)
    point_xy: Tuple[float, float], # (x,y)
    R: float,
    *,
    tree=None,                     # cKDTree or None
    return_dist: bool = False,
) -> List[int] | Tuple[List[int], np.ndarray]:
    """
    점 좌표 (x,y)를 입력받아 반경 R 이내의 모든 mesh '정점(vertex) 인덱스'를 리턴한다.

    - tree(cKDTree)가 주어지면 O(log N + k)로 빠르게 검색
    - tree가 없으면 브루트포스(O(N))로 계산

    Returns:
      idx_list (sorted)
      (optional) dists: idx_list와 같은 순서의 거리 배열
    """
    if R < 0:
        raise ValueError("R must be non-negative.")
    vertices = np.asarray(vertices, dtype=np.float64)
    if vertices.ndim != 2 or vertices.shape[1] != 2:
        raise ValueError("vertices must be (N,2).")

    x, y = float(point_xy[0]), float(point_xy[1])

    if tree is not None:
        idx = tree.query_ball_point([x, y], r=float(R))
        idx = np.array(idx, dtype=np.int64)
        if idx.size == 0:
            return ([], np.array([], dtype=np.float64)) if return_dist else []
        pts = vertices[idx]
        d = np.hypot(pts[:, 0] - x, pts[:, 1] - y)
        order = np.argsort(d)
        idx_sorted = idx[order].tolist()
        if return_dist:
            return idx_sorted, d[order]
        return idx_sorted

    # fallback: brute force
    dx = vertices[:, 0] - x
    dy = vertices[:, 1] - y
    d = np.hypot(dx, dy)
    mask = d <= float(R)
    idx = np.nonzero(mask)[0]
    if idx.size == 0:
        return ([], np.array([], dtype=np.float64)) if return_dist else []
    order = np.argsort(d[idx])
    idx_sorted = idx[order].tolist()
    if return_dist:
        return idx_sorted, d[idx][order]
    return idx_sorted

from scipy.spatial import cKDTree

def fine_points_within_R_of_path(
    coarse_vertices: np.ndarray,   # (Nc,2)
    fine_vertices: np.ndarray,     # (Nf,2)
    path_nodes: List[int],         # indices into coarse_vertices
    R: float,
) -> Tuple[List[np.ndarray], np.ndarray]:
    """
    For each path node (in coarse mesh), return indices of fine points within radius R.
    Also returns the union of all such indices (unique sorted).
    """
    if R < 0:
        raise ValueError("R must be non-negative.")
    if coarse_vertices.ndim != 2 or coarse_vertices.shape[1] != 2:
        raise ValueError("coarse_vertices must be (Nc,2).")
    if fine_vertices.ndim != 2 or fine_vertices.shape[1] != 2:
        raise ValueError("fine_vertices must be (Nf,2).")

    path_nodes = np.asarray(path_nodes, dtype=np.int64)
    if path_nodes.size == 0:
        return [], np.array([], dtype=np.int64)

    Q = coarse_vertices[path_nodes]  # query points (P,2)

    tree = cKDTree(fine_vertices)
    hits_list = tree.query_ball_point(Q, r=float(R))  # list[list[int]]

    hits_per = [np.asarray(h, dtype=np.int64) for h in hits_list]

    if any(len(h) for h in hits_per):
        hits_union = np.unique(np.concatenate(hits_per))
    else:
        hits_union = np.array([], dtype=np.int64)

    return fine_vertices[hits_union], hits_union

def make_submesh_points_and_adjacency(
    vertices: np.ndarray,        # (N,2) global vertices
    adj_fixed: np.ndarray,       # (N,K) global fixed adjacency (neighbor or -1)
    union_nodes: List[int],      # global node indices to keep
    *,
    missing: int = -1,
    sort_nodes: bool = True,
) -> Tuple[np.ndarray, np.ndarray, List[int], Dict[int, int]]:
    """
    union_nodes로 induced subgraph를 만들고, 로컬 인덱스로 재매핑한다.
    '고정된 k 슬롯'을 유지하기 위해 ragged가 아니라 (M,K) fixed adjacency를 만든다.

    Returns:
      sub_points : (M,2) float
      sub_adj    : (M,K) int32  (로컬 인덱스 기준, 없는 이웃은 missing)
      sub_nodes  : List[int]    (local idx -> global idx)
      old2new    : Dict[int,int](global idx -> local idx)
    """
    if vertices.ndim != 2 or vertices.shape[1] != 2:
        raise ValueError("vertices must be (N,2).")
    vertices = np.asarray(vertices)
    N = vertices.shape[0]

    adj_fixed = np.asarray(adj_fixed, dtype=np.int32)
    if adj_fixed.ndim != 2 or adj_fixed.shape[0] != N:
        raise ValueError("adj_fixed must be (N,K) with N matching vertices.")
    K = int(adj_fixed.shape[1])

    # 1) unique + optional sort
    sub_nodes = list({int(v) for v in union_nodes})
    if sort_nodes:
        sub_nodes.sort()

    # 2) global->local
    old2new: Dict[int, int] = {v: i for i, v in enumerate(sub_nodes)}
    sub_set = set(sub_nodes)

    # 3) points (local order)
    sub_points = vertices[np.array(sub_nodes, dtype=np.int64)].copy()

    # 4) fixed adjacency remap, preserving k slots
    M = len(sub_nodes)
    sub_adj = np.full((M, K), missing, dtype=np.int32)

    for v_old in sub_nodes:
        v_new = old2new[v_old]
        row = adj_fixed[v_old]  # (K,)
        for k in range(K):
            u_old = int(row[k])
            if u_old == missing:
                continue
            if u_old in sub_set:
                sub_adj[v_new, k] = int(old2new[u_old])
            # else: keep missing, slot preserved

    return sub_points, sub_adj, sub_nodes, old2new

def dijkstra_turn_state_backward_kin_all(
    cache,                      # EdgeCacheFixed (must have adj, edge_id, in_edge_id, dst_k_in, N, K, missing)
    base_cost_e: np.ndarray,     # (E,)
    theta_e_deg: np.ndarray,     # (E,)
    goal: int,
    goal_k_in: Optional[int] = None,   # None이면 goal의 모든 k_in(유효한 것들)을 멀티소스
    *,
    th1: float = 40.0, lam1: float = 10.0,
    th2: float = 50.0, lam2: float = np.inf,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Backward Dijkstra on fixed-slot state (v, k_in).

    State meaning:
      (v, k_in) means we entered v from p = cache.adj[v, k_in] (p -> v).

    dist_b_in[v, k_in] = minimal remaining cost from that state to reach the goal.

    Returns:
      dist_b_in: (N,K) float (inf if unreachable)
      next_node: (N,K) int32  forward pointer: from (p,k_in_p) go next to node v
      next_k_in: (N,K) int32  forward pointer: next state's k_in at that next node
    """
    N = int(cache.N)
    K = int(cache.K)
    missing = int(getattr(cache, "missing", -1))

    if base_cost_e.shape[0] != cache.E or theta_e_deg.shape[0] != cache.E:
        raise ValueError("base_cost_e/theta_e_deg must have shape (E,) matching cache.E.")

    goal = int(goal)
    if goal < 0 or goal >= N:
        raise ValueError("goal out of range")

    valid_in = (cache.adj != missing)  # (N,K) : 유효한 k_in 슬롯

    INF = float("inf")
    dist_b_in = np.full((N, K), INF, dtype=float)
    next_node = np.full((N, K), -1, dtype=np.int32)
    next_k_in = np.full((N, K), -1, dtype=np.int32)

    h: List[Tuple[float, int, int]] = []  # (d, v, k_in)

    # --- initialize goal terminal states ---
    if not bool(valid_in[goal].any()):
        # goal이 고립(유효 k_in 상태가 없음)
        return dist_b_in, next_node, next_k_in

    if goal_k_in is None:
        # goal의 모든 "유효한" k_in을 멀티소스
        for k_in in range(K):
            if not bool(valid_in[goal, k_in]):
                continue
            dist_b_in[goal, k_in] = 0.0
            heapq.heappush(h, (0.0, goal, k_in))
    else:
        goal_k_in = int(goal_k_in)
        if goal_k_in < 0 or goal_k_in >= K or not bool(valid_in[goal, goal_k_in]):
            raise ValueError(f"goal_k_in invalid: {goal_k_in} for goal={goal}")
        dist_b_in[goal, goal_k_in] = 0.0
        heapq.heappush(h, (0.0, goal, goal_k_in))

    # --- backward dijkstra ---
    while h:
        d, v, k_in_v = heapq.heappop(h)
        if d != dist_b_in[v, k_in_v]:
            continue

        # (v, k_in_v) means: p -> v where p is the neighbor at this slot
        p = int(cache.adj[v, k_in_v])
        if p == missing:
            continue

        # edge id for (p -> v) is directly available for this state
        e_pv = int(cache.in_edge_id[v, k_in_v])
        if e_pv == missing:
            continue

        theta_out = float(theta_e_deg[e_pv])   # p -> v heading (outgoing from p)
        c_out = float(base_cost_e[e_pv])
        if not np.isfinite(theta_out) or not np.isfinite(c_out):
            continue

        # predecessor states are (p, k_in_p) for all valid k_in_p (q -> p)
        for k_in_p in range(K):
            if not bool(valid_in[p, k_in_p]):
                continue

            e_qp = int(cache.in_edge_id[p, k_in_p])  # q -> p
            if e_qp == missing:
                continue

            theta_in = float(theta_e_deg[e_qp])      # heading into p
            if not np.isfinite(theta_in):
                continue

            delta = wrap180_deg(theta_out - theta_in)
            w = c_out + turn_penalty_deg(delta, th1, lam1, th2, lam2)
            nd = float(d) + float(w)

            if nd < dist_b_in[p, k_in_p]:
                dist_b_in[p, k_in_p] = nd
                # forward pointers: from (p,k_in_p) the next state is (v,k_in_v)
                next_node[p, k_in_p] = int(v)
                next_k_in[p, k_in_p] = int(k_in_v)
                heapq.heappush(h, (nd, p, k_in_p))

    return dist_b_in, next_node, next_k_in

def reconstruct_path_from_next_kin(
    cache,                           # EdgeCacheFixed (adj, edge_id, N, K, missing)
    next_node: np.ndarray,           # (N,K) int32 : next_node[v,k_in] = next v
    next_k_in: np.ndarray,           # (N,K) int32 : next_k_in[v,k_in] = next k_in at that next v
    start: int,
    goal: int,
    start_k_in: int,                 # 시작 상태 (start, start_k_in)
    *,
    max_steps: Optional[int] = None,
) -> Tuple[List[int], List[int], List[int]]:
    """
    Reconstruct forward path to goal using outputs of dijkstra_turn_state_backward_kin_all_fixed.

    State is (v, k_in) where k_in is fixed slot index telling which neighbor we came FROM.

    Returns:
      node_path: [start, ..., goal]
      edge_path: directed edge ids along node_path
      k_in_path: k_in indices along visited states (start 포함, goal 포함)
    """
    start = int(start)
    goal = int(goal)
    v = int(start)
    k_in = int(start_k_in)

    N = int(cache.N)
    K = int(cache.K)
    missing = int(getattr(cache, "missing", -1))

    if start == goal:
        return [start], [], [k_in]

    if max_steps is None:
        max_steps = 10 * max(1, N)

    # 유효성 체크
    if v < 0 or v >= N:
        raise ValueError("start out of range")
    if k_in < 0 or k_in >= K:
        raise ValueError(f"start_k_in out of range: k_in={k_in}, K={K}")
    if int(cache.adj[v, k_in]) == missing:
        raise ValueError(f"start state (v={v}, k_in={k_in}) is invalid: cache.adj is missing.")

    node_path: List[int] = [v]
    k_in_path: List[int] = [k_in]
    edge_path: List[int] = []

    for _ in range(int(max_steps)):
        vn = int(next_node[v, k_in])
        kn = int(next_k_in[v, k_in])

        # 도달 불가
        if vn < 0 or vn >= N:
            return [], [], []

        if kn < 0 or kn >= K:
            raise RuntimeError(f"next_k_in out of range at next v={vn}: kn={kn}, K={K}")

        # edge id는 fixed 슬롯에서 바로 얻을 수 있음:
        # 현재 노드 v에서 vn으로 가는 슬롯 k_out을 찾아야 하는데,
        # backward pointer가 주는 건 (vn, kn) (즉 vn의 k_in)이라서,
        # 간단히 edge_id[v, k_out]를 찾는 대신 src/dst 스캔을 하지 않으려면
        # "v->vn"의 k_out을 미리 알아야 함.
        #
        # 가장 안전한 방법(추가 필드 없이): K가 작으니 v의 슬롯을 훑어서 vn을 찾는다.
        k_out = -1
        for kk in range(K):
            if int(cache.adj[v, kk]) == vn:
                k_out = kk
                break
        if k_out < 0:
            raise RuntimeError(f"Cannot find slot k_out at v={v} leading to vn={vn} in cache.adj.")

        eid = int(cache.edge_id[v, k_out])
        if eid == missing:
            raise RuntimeError(f"Edge id missing for v={v}, k_out={k_out} (to vn={vn}).")

        edge_path.append(eid)
        node_path.append(vn)
        k_in_path.append(kn)

        if vn == goal:
            return node_path, edge_path, k_in_path

        v, k_in = vn, kn

        if int(cache.adj[v, k_in]) == missing:
            raise RuntimeError(f"Invalid state reached: (v={v}, k_in={k_in}) has missing predecessor slot.")

    return [], [], []

def k_from_coordinate(sx: float, sy: float, fx: float, fy: float, mod,neigh_mod) -> int:
    dx = fx - sx
    dy = fy - sy

    if dx == 0 and dy == 0:
        raise ValueError("start와 finish가 동일하면 방향 각도를 정의할 수 없다.")

    # y축 기준 각도 (deg) in [0, 360)
    theta = math.degrees(math.atan2(dx, dy)) % 360.0
    if mod=='Hexa':
        if neigh_mod == 'Edges':
            return((theta-30)//60+1)%6
        if neigh_mod == 'Extended_edges':
            return ((theta-15)//30+1)%12
        if neigh_mod == 'Extra_extended_edges':
            return ((theta-7.5)//15+1)%24
    if mod=='Square':
        if neigh_mod == 'Edges':
            return((theta-45)//90+1)%4
        if neigh_mod == 'Extended_edges':
            return ((theta-22.5)//45+1)%8
        if neigh_mod == 'Extra_extended_edges':
            return ((theta-11.25)//22.5+1)%16

def cal_backward(points,way_points,cache,cost_e,theta_deg_e,idx_list,fixed_k_list=None):
    node_path_lists=[]
    edge_path_lists=[]
    dist_lists=[]
    sum=0
    for point_index in range(len(way_points)-1):
        dist_b_in, next_node, next_k_in = dijkstra_turn_state_backward_kin_all(
            cache=cache,
            base_cost_e=cost_e,  # (E,)
            theta_e_deg=theta_deg_e,  # (E,)
            goal=idx_list[point_index+1],
            goal_k_in=fixed_k_list[point_index+1],  # goal에서 "어느 이웃으로 나가는지" (out-slot)
            th1=40, lam1=10,
            th2=50.0, lam2=np.inf,
        )
        node_path, edge_path, k_in_path = reconstruct_path_from_next_kin(
            cache=cache,
            next_node=next_node,  # next_node[v][k_out] = 다음 노드 v_next
            next_k_in=next_k_in,  # next_k[v][k_out] = 다음 노드에서의 k_out
            start=idx_list[point_index],
            goal=idx_list[point_index+1],
            start_k_in=fixed_k_list[point_index])
        print(dist_b_in[int(idx_list[point_index])][int(fixed_k_list[point_index])])
        sum+=dist_b_in[int(idx_list[point_index])][int(fixed_k_list[point_index])]
        node_path_lists.append(node_path)
        edge_path_lists.append(edge_path)
        dist_lists.append(dist_b_in)
    print(sum)
    return dist_lists,node_path_lists, edge_path_lists



def calculate_path_and_time(way_points,cache,cost_e,theta_deg_e,idx_list,mode,prev_heading=0.0,fixed_k_list=None):

    if mode=='greedy':
        terminal_mod='goal_best'
        use_k_boolean=False
        k_in=0
    if mode=='chromosome':
        terminal_mod='goal_allk'
        use_k_boolean=True

    fitness=0
    node_path_list = []
    edge_path_list = []
    print(len(way_points))
    for point_index in range(len(way_points) - 1):
        if mode=='chromosome':
            k_in=fixed_k_list[point_index]
        dist, prev_node, prev_k, best_goal, best_goal_k = dijkstra_turn_state_core(
            cache=cache,
            base_cost_e=cost_e,  # (E,)
            theta_e_deg=theta_deg_e,  # (E,)  (+y clockwise, deg)
            start=idx_list[point_index],
            goal=idx_list[point_index + 1],

            # turn penalty
            th1=40.0, lam1=10.0,
            th2=50.0, lam2=np.inf,

            # start heading (optional)
            use_start_heading=True,
            start_heading_deg=prev_heading,

            # termination mode
            termination=terminal_mod)  # "goal": goal pop 시 종료, "all": PQ empty까지)

        node_path, edge_path, k_path = reconstruct_path_from_prev(
            cache,
            prev_node,  # (N,n)
            prev_k,  # (N,n)
            start=idx_list[point_index],
            goal=idx_list[point_index + 1],
            best_goal_k=best_goal_k,
            use_k=use_k_boolean,
            fixed_k=k_in,
        )
        v_goal = idx_list[point_index + 1]
        if mode=='greedy':
            k_in=best_goal_k
        fitness+=dist[idx_list[point_index + 1]][k_in]
        eid = int(cache.in_edge_id[v_goal, k_in])
        theta_in_goal = float(theta_deg_e[eid])
        prev_heading = theta_in_goal
        node_path_list.append(node_path)
        edge_path_list.append(edge_path)

    return fitness, node_path_list, edge_path_list


import numpy as np
import matplotlib.pyplot as plt
from typing import List, Optional, Sequence, Dict, Any, Tuple

def current_uv_from_speed_angle(
    speed: np.ndarray,        # (N,)
    angle_deg: np.ndarray,    # (N,) 0=+y, 90=+x (clockwise from +y)
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Convert (speed, angle_deg) to (u,v) for quiver.
    Here:
      angle=0 => +y, angle=90 => +x
    so:
      u = speed * sin(theta), v = speed * cos(theta)
    """
    th = np.deg2rad(angle_deg.astype(float))
    u = speed.astype(float) * np.sin(th)
    v = speed.astype(float) * np.cos(th)
    return u, v


def edge_path_to_xy(
    points: np.ndarray,   # (N,2)
    cache,                # EdgeCacheFixed (src,dst)
    edge_path: Sequence[int],
) -> np.ndarray:
    """
    Build polyline coordinates (M,2) following the edge_path.
    Returns a sequence of points along the path (node coords in order).
    """
    if len(edge_path) == 0:
        return np.zeros((0, 2), dtype=float)

    edge_path = [int(e) for e in edge_path]
    src0 = int(cache.src[edge_path[0]])
    coords = [points[src0]]

    for e in edge_path:
        t = int(cache.dst[int(e)])
        coords.append(points[t])

    return np.asarray(coords, dtype=float)


def plot_paths_with_currents(
    path_pack_list: List[Dict[str, Any]],
    *,
    show_mesh_points: bool = True,
    mesh_point_size: float = 6.0,
    show_currents: bool = True,
    current_stride: Optional[int] = None,   # None이면 자동
    max_arrows: int = 1500,                 # 너무 많으면 자동 다운샘플
    arrow_scale: Optional[float] = None,    # None이면 matplotlib 기본 스케일
    arrow_width: float = 0.002,
    show_start_end: bool = True,
    equal_aspect: bool = True,
    title: Optional[str] = None,
):
    """
    path_pack_list: 경로별로 아래 키들을 가진 dict 리스트
      required:
        - "points": (Ni,2) sub_points
        - "cache":  EdgeCacheFixed
        - "edge_path": List[int] (최종 경로 edge id 리스트; cache 기준)
      optional (해류 표기용):
        - "current_speed": (Ni,)
        - "current_angle_deg": (Ni,) 0=+y clockwise
      optional (라벨):
        - "label": str

    한 figure에 여러 경로를 overlay로 그린다.
    """
    fig, ax = plt.subplots()

    for idx, pack in enumerate(path_pack_list):
        pts = np.asarray(pack["points"], dtype=float)
        cache = pack["cache"]
        edge_path = pack["edge_path"]
        label = pack.get("label", f"path{idx}")

        # 1) mesh points
        if show_mesh_points:
            ax.scatter(pts[:, 0], pts[:, 1], s=mesh_point_size)

        # 2) currents (quiver)
        if show_currents and ("current_speed" in pack) and ("current_angle_deg" in pack):
            spd = np.asarray(pack["current_speed"], dtype=float)
            ang = np.asarray(pack["current_angle_deg"], dtype=float)
            u, v = current_uv_from_speed_angle(spd, ang)

            N = pts.shape[0]
            # 자동 stride
            stride = current_stride
            if stride is None:
                if N > max_arrows:
                    stride = int(np.ceil(N / max_arrows))
                else:
                    stride = 1

            sl = slice(0, N, stride)
            ax.quiver(
                pts[sl, 0], pts[sl, 1],
                u[sl], v[sl],
                angles="xy",
                scale=arrow_scale,     # None이면 default
                width=arrow_width,
            )

        # 3) path polyline from edge_path
        poly = edge_path_to_xy(pts, cache, edge_path)
        if poly.shape[0] >= 2:
            ax.plot(poly[:, 0], poly[:, 1], linewidth=2.0, label=label)

            if show_start_end:
                ax.scatter(poly[0, 0], poly[0, 1], s=40.0)      # start
                ax.scatter(poly[-1, 0], poly[-1, 1], s=40.0)    # end

    if title is not None:
        ax.set_title(title)

    ax.legend()
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    if equal_aspect:
        ax.set_aspect("equal", adjustable="box")
    plt.show()





def _uv_from_speed_angle_ycw(speed: np.ndarray, angle_deg: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    angle_deg: 0=+y, 90=+x (clockwise from +y)
    return: (ux, uy) for quiver (x,y plane)
    """
    th = np.deg2rad(angle_deg.astype(float))
    ux = speed.astype(float) * np.sin(th)
    uy = speed.astype(float) * np.cos(th)
    return ux, uy


def _draw_waypoint_radius_circles(
    waypoints_xy: np.ndarray,
    *,
    radius: float = 60.0,
    face_color: str = "#1f77b4",   # matplotlib default blue 느낌
    alpha: float = 0.18,
    edge_color: Optional[str] = None,
    edge_alpha: float = 0.0,
    zorder: int = 19,
):
    """
    Draw translucent circles of given radius around each waypoint.
    """
    wp = np.asarray(waypoints_xy, dtype=float)
    if wp.ndim != 2 or wp.shape[1] != 2:
        raise ValueError("waypoints_xy must be (M,2).")

    ax = plt.gca()
    for x, y in wp:
        circ = plt.Circle(
            (float(x), float(y)),
            float(radius),
            facecolor=face_color,
            alpha=float(alpha),
            edgecolor=edge_color if edge_color is not None else "none",
            linewidth=1.0 if edge_color is not None else 0.0,
        )
        circ.set_zorder(zorder)
        if edge_color is not None:
            circ.set_alpha(alpha)  # face alpha
            # edge alpha는 따로 컨트롤하고 싶으면 edgecolor에 RGBA를 넣는 방식이 필요함
        ax.add_patch(circ)

def plot_packs_with_connectors_and_waypoints(
    path_pack_list: List[Dict[str, Any]],
    *,
    # --- global background current (optional) ---
    global_vertices: Optional[np.ndarray] = None,     # (Ng,2)
    global_curr_ux: Optional[np.ndarray] = None,      # (Ng,)
    global_curr_uy: Optional[np.ndarray] = None,      # (Ng,)
    global_curr_v: Optional[np.ndarray] = None,       # (Ng,)
    global_curr_theta_deg: Optional[np.ndarray] = None,# (Ng,)
    show_global_current: bool = False,
    global_current_stride: int = 1,
    global_current_color: str = "#0033cc",
    global_current_alpha: float = 0.55,
    global_current_width: float = 0.002,

    # --- per-pack current (optional) ---
    show_current_quiver: bool = False,
    current_stride: int = 1,
    current_color: str = "#0033cc",
    current_alpha: float = 0.9,
    current_width: float = 0.003,

    # --- draw graph ---
    draw_all_edges: bool = False,
    all_edges_stride: int = 1,
    edge_color: str = "black",
    all_edge_alpha: float = 0.20,
    all_edge_lw: float = 0.6,

    # --- path style ---
    path_alpha: float = 0.95,
    path_lw: float = 2.5,

    # --- theta on path ---
    show_theta_quiver: bool = True,
    theta_on: str = "src",        # "src" or "mid"
    theta_arrow_len: float = 0.15,
    show_theta_text: bool = False,

    # --- connector points between packs (yellow) ---
    show_connector_points: bool = True,
    connector_color: str = "yellow",
    connector_size: float = 10,
    connector_marker: str = "o",

    # --- original waypoints in world coords (red) ---
    waypoints_xy: Optional[np.ndarray] = None,        # (M,2) in same coordinate system as vertices
    show_waypoints: bool = True,
    waypoint_color: str = "red",
    waypoint_size: float = 10,
    waypoint_marker: str = "o",

    # --- only show global start/end markers ---
    show_global_start_end: bool = True,
    start_marker: str = "o",
    end_marker: str = "X",
    start_end_size: float = 10,

    figsize=(10, 8),
    dpi=150,
):
    if len(path_pack_list) == 0:
        raise ValueError("path_pack_list is empty.")

    plt.figure(figsize=figsize, dpi=dpi)

    # -----------------------------
    # (0) global current field (behind everything)
    # -----------------------------
    def _uv_from_speed_angle_ycw(speed: np.ndarray, angle_deg: np.ndarray):
        th = np.deg2rad(angle_deg.astype(float))
        ux = speed.astype(float) * np.sin(th)
        uy = speed.astype(float) * np.cos(th)
        return ux, uy

    if show_global_current:
        if global_vertices is None:
            raise ValueError("global_vertices is required when show_global_current=True.")
        gv = np.asarray(global_vertices, dtype=float)
        gx, gy = gv[:, 0], gv[:, 1]
        stride = max(1, int(global_current_stride))
        idx = np.arange(0, gv.shape[0], stride)

        if global_curr_ux is not None and global_curr_uy is not None:
            ux = np.asarray(global_curr_ux, dtype=float)[idx]
            uy = np.asarray(global_curr_uy, dtype=float)[idx]
        elif global_curr_v is not None and global_curr_theta_deg is not None:
            spd = np.asarray(global_curr_v, dtype=float)
            ang = np.asarray(global_curr_theta_deg, dtype=float)
            ux_all, uy_all = _uv_from_speed_angle_ycw(spd, ang)
            ux = ux_all[idx]
            uy = uy_all[idx]
        else:
            raise ValueError("Provide either (global_curr_ux, global_curr_uy) or (global_curr_v, global_curr_theta_deg).")

        plt.quiver(
            gx[idx], gy[idx], ux, uy,
            color=global_current_color,
        )

    # -----------------------------
    # (A) gather connector points (between packs) and global start/end
    # -----------------------------
    def _pack_start_end_xy(pack: Dict[str, Any]) -> Tuple[np.ndarray, np.ndarray]:
        pts = np.asarray(pack["points"], dtype=float)
        cache = pack["cache"]
        edge_path = pack.get("edge_path", None)
        node_path = pack.get("node_path", None)

        if edge_path is not None and len(edge_path) > 0:
            # edge_path must be a flat list for a pack (your use case)
            e0 = int(edge_path[0])
            eL = int(edge_path[-1])
            s = int(cache.src[e0])
            t = int(cache.dst[eL])
            return pts[s], pts[t]

        if node_path is not None and len(node_path) > 0:
            s = int(node_path[0])
            t = int(node_path[-1])
            return pts[s], pts[t]

        raise ValueError("Each pack must have edge_path or node_path.")

    pack_starts = []
    pack_ends = []
    for pack in path_pack_list:
        sxy, exy = _pack_start_end_xy(pack)
        pack_starts.append(sxy)
        pack_ends.append(exy)

    pack_starts = np.asarray(pack_starts, dtype=float)
    pack_ends = np.asarray(pack_ends, dtype=float)

    global_start_xy = pack_starts[0]
    global_end_xy = pack_ends[-1]

    # connector points = end of pack i (i=0..P-2)
    # (start of next pack is basically same location; we mark only once)
    connector_xy = pack_ends[:-1].copy()  # (P-1,2)

    # -----------------------------
    # (B) draw each pack (currents, edges, paths, theta)
    #     BUT: do NOT draw per-pack start/end markers
    # -----------------------------
    for pack in path_pack_list:
        vertices = np.asarray(pack["points"], dtype=float)
        cache = pack["cache"]
        x = vertices[:, 0]
        y = vertices[:, 1]

        # per-pack current
        if show_current_quiver:
            N = vertices.shape[0]
            stride = max(1, int(current_stride))
            idx = np.arange(0, N, stride)

            if "curr_ux" in pack and "curr_uy" in pack and pack["curr_ux"] is not None and pack["curr_uy"] is not None:
                ux = np.asarray(pack["curr_ux"], dtype=float)[idx]
                uy = np.asarray(pack["curr_uy"], dtype=float)[idx]
            elif "current_speed" in pack and "current_angle_deg" in pack:
                spd = np.asarray(pack["current_speed"], dtype=float)
                ang = np.asarray(pack["current_angle_deg"], dtype=float)
                ux_all, uy_all = _uv_from_speed_angle_ycw(spd, ang)
                ux = ux_all[idx]
                uy = uy_all[idx]
            else:
                ux = uy = None

            if ux is not None:
                plt.quiver(
                    x[idx], y[idx], ux, uy,
                    color=current_color,
                    alpha=current_alpha,
                    width=current_width,
                    zorder=2,
                )

        # all edges
        if draw_all_edges:
            E = int(cache.E)
            step = max(1, int(all_edges_stride))
            for e in range(0, E, step):
                u = int(cache.src[e])
                v = int(cache.dst[e])
                plt.plot([x[u], x[v]], [y[u], y[v]],
                         color=edge_color, alpha=all_edge_alpha,
                         linewidth=all_edge_lw, zorder=3)

        # path
        edge_path = pack.get("edge_path", None)
        node_path = pack.get("node_path", None)

        if edge_path is not None and len(edge_path) > 0:
            for e in edge_path:
                e = int(e)
                u = int(cache.src[e])
                v = int(cache.dst[e])
                plt.plot([x[u], x[v]], [y[u], y[v]],
                         color=edge_color, alpha=path_alpha, linewidth=path_lw, zorder=5)

            # theta arrows
            if show_theta_quiver:
                theta_deg_e = pack.get("theta_deg_e", None)
                if theta_deg_e is None:
                    raise ValueError("pack['theta_deg_e'] is required when show_theta_quiver=True and edge_path is provided.")

                pxs, pys, ux_list, uy_list = [], [], [], []
                for e in edge_path:
                    e = int(e)
                    th = float(theta_deg_e[e])
                    if not np.isfinite(th):
                        continue
                    u0 = int(cache.src[e])
                    v0 = int(cache.dst[e])
                    if theta_on == "src":
                        px, py = x[u0], y[u0]
                    else:
                        px, py = 0.5 * (x[u0] + x[v0]), 0.5 * (y[u0] + y[v0])

                    th_rad = np.deg2rad(th)
                    dir_x = np.sin(th_rad)
                    dir_y = np.cos(th_rad)

                    pxs.append(px); pys.append(py)
                    ux_list.append(dir_x); uy_list.append(dir_y)

                    if show_theta_text:
                        plt.text(px, py, f"{th:.1f}°", fontsize=8, ha="left", va="bottom",
                                 color="black", zorder=9)

                if len(pxs) > 0:
                    plt.quiver(
                        np.asarray(pxs), np.asarray(pys),
                        np.asarray(ux_list) * theta_arrow_len,
                        np.asarray(uy_list) * theta_arrow_len,
                        angles="xy", scale_units="xy", scale=1.0,
                        width=0.003,
                        color="black", alpha=path_alpha, zorder=8
                    )

        elif node_path is not None and len(node_path) > 0:
            pts = np.asarray(node_path, dtype=int)
            plt.plot(x[pts], y[pts], color=edge_color, alpha=path_alpha, linewidth=path_lw, zorder=5)
        else:
            continue

    # -----------------------------
    # (C) overlay: connector points (yellow) and original waypoints (red)
    # -----------------------------
    if show_connector_points and connector_xy.shape[0] > 0:
        plt.scatter(
            connector_xy[:, 0], connector_xy[:, 1],
            s=connector_size, marker=connector_marker,
            color=connector_color, edgecolors="none",
            zorder=20
        )

    if show_waypoints and waypoints_xy is not None:
        wp = np.asarray(waypoints_xy, dtype=float)
        if wp.ndim != 2 or wp.shape[1] != 2:
            raise ValueError("waypoints_xy must be (M,2).")
        plt.scatter(
            wp[:, 0], wp[:, 1],
            s=waypoint_size, marker=waypoint_marker,
            color=waypoint_color, edgecolors="none",
            zorder=21
        )
        _draw_waypoint_radius_circles(
            waypoints_xy,
            radius=90.0,
            face_color="#1f77b4",
            alpha=0.18,
            zorder=19,
        )

    # -----------------------------
    # (D) only global start/end markers
    # -----------------------------
    if show_global_start_end:
        plt.scatter([global_start_xy[0]], [global_start_xy[1]],
                    s=start_end_size, marker=start_marker, color=edge_color, zorder=30)
        plt.scatter([global_end_xy[0]], [global_end_xy[1]],
                    s=start_end_size, marker=end_marker, color=edge_color, zorder=30)

    plt.gca().set_aspect("equal", adjustable="box")
    plt.grid(True, alpha=0.2)
    plt.show()

In [14]:

import random
import numpy as np

import time
from typing import Any, List, Tuple, Optional


In [15]:
@dataclass
class PatrolRegion:
    region_id: int
    x0: float
    x1: float
    y0: float
    y1: float


def make_8_regions(x_s: float, x_f: float, y_s: float, y_f: float) -> List[PatrolRegion]:
    """
    전체 영역을 4 x 2 = 8개로 분할
    각 영역 크기: 900 x 900 (현재 값 기준)
    """
    W = (x_f - x_s) / 4.0
    H = (y_f - y_s) / 2.0

    regions = []
    rid = 0
    for row in range(2):         # y direction
        for col in range(4):     # x direction
            rx0 = x_s + col * W
            rx1 = x_s + (col + 1) * W
            ry0 = y_s + row * H
            ry1 = y_s + (row + 1) * H
            regions.append(PatrolRegion(rid, rx0, rx1, ry0, ry1))
            rid += 1
    return regions


# ---------------------------------------------------
# 2) 영역 내부 4개 waypoint 생성
# ---------------------------------------------------
def make_region_waypoints(region: PatrolRegion, margin: float = 180.0) -> np.ndarray:
    """
    각 영역 안에 4개 순찰 지점 생성
    순서: 좌하 -> 우하 -> 우상 -> 좌상
    """
    x0, x1, y0, y1 = region.x0, region.x1, region.y0, region.y1

    if (x1 - x0) <= 2 * margin or (y1 - y0) <= 2 * margin:
        raise ValueError("margin too large for region size.")

    pts = np.array([
        [x0 + margin, y0 + margin],  # bottom-left
        [x1 - margin, y0 + margin],  # bottom-right
        [x1 - margin, y1 - margin],  # top-right
        [x0 + margin, y1 - margin],  # top-left
    ], dtype=float)
    return pts


# ---------------------------------------------------
# 3) 좌표 -> 가장 가까운 vertex 찾기
# ---------------------------------------------------
def nearest_vertex_index(points: np.ndarray, xy: np.ndarray) -> int:
    """
    단순 최근접 vertex index
    """
    diff = points - xy[None, :]
    d2 = np.sum(diff * diff, axis=1)
    return int(np.argmin(d2))


def nearest_free_vertex_index_in_region(
    points: np.ndarray,
    xy: np.ndarray,
    region: PatrolRegion,
    blocked_mask: np.ndarray,
) -> int:
    """
    region 내부이면서 blocked가 아닌 free vertex 중에서
    xy와 가장 가까운 vertex index 반환
    """
    mask = (
        (points[:, 0] >= region.x0) & (points[:, 0] <= region.x1) &
        (points[:, 1] >= region.y0) & (points[:, 1] <= region.y1) &
        (~blocked_mask)
    )

    idxs = np.where(mask)[0]
    if len(idxs) == 0:
        raise RuntimeError(f"No free vertex exists in region {region.region_id}")

    sub = points[idxs]
    diff = sub - xy[None, :]
    d2 = np.sum(diff * diff, axis=1)
    return int(idxs[np.argmin(d2)])


# ---------------------------------------------------
# 4) waypoint 좌표들을 vertex index로 변환
# ---------------------------------------------------
def waypoint_vertices_for_region(
    points: np.ndarray,
    region: PatrolRegion,
    blocked_mask: np.ndarray,
    margin: float = 180.0,
) -> Tuple[np.ndarray, List[int]]:
    wp_xy_target = make_region_waypoints(region, margin=margin)

    wp_vids = [
        nearest_free_vertex_index_in_region(points, wp, region, blocked_mask)
        for wp in wp_xy_target
    ]

    # 실제 free vertex 좌표로 waypoint 위치를 덮어씀
    wp_xy = points[np.array(wp_vids)].copy()

    return wp_xy, wp_vids

# ---------------------------------------------------
# 5) 두 정점 사이 경로 계산
# ---------------------------------------------------
def shortest_path_between_vertices(
    cache: EdgeCache,
    cost_e: np.ndarray,
    theta_e_deg: np.ndarray,
    start_vid: int,
    goal_vid: int,
    *,
    th1: float = 40.0,
    lam1: float = 10.0,
    th2: float = 50.0,
    lam2: float = np.inf,
    use_start_heading: bool = False,
    start_heading_deg: float = 0.0,
    termination: Literal["goal_best", "goal_allk", "all"] = "goal_best",
):
    dist, prev_node, prev_k, best_goal, best_goal_k = dijkstra_turn_state_core(
        cache=cache,
        base_cost_e=cost_e,
        theta_e_deg=theta_e_deg,
        start=start_vid,
        goal=goal_vid,
        th1=th1, lam1=lam1,
        th2=th2, lam2=lam2,
        use_start_heading=use_start_heading,
        start_heading_deg=start_heading_deg,
        termination=termination,
    )

    node_path, edge_path, k_path = reconstruct_path_from_prev(
        cache=cache,
        prev_node=prev_node,
        prev_k=prev_k,
        start=start_vid,
        goal=goal_vid,
        best_goal_k=best_goal_k,
    )

    return {
        "dist": dist,
        "prev_node": prev_node,
        "prev_k": prev_k,
        "best_goal": best_goal,
        "best_goal_k": best_goal_k,
        "node_path": node_path,
        "edge_path": edge_path,
        "k_path": k_path,
    }


# ---------------------------------------------------
# 6) 한 영역의 4개 waypoint 순찰 루프 생성
# ---------------------------------------------------
def build_patrol_loop_for_region(
    points: np.ndarray,
    cache: EdgeCache,
    cost_e: np.ndarray,
    theta_e_deg: np.ndarray,
    region: PatrolRegion,
    blocked_mask: np.ndarray,
    *,
    margin: float = 180.0,
    close_loop: bool = True,
    th1: float = 40.0,
    lam1: float = 10.0,
    th2: float = 50.0,
    lam2: float = np.inf,
) -> Dict:
    wp_xy, wp_vids = waypoint_vertices_for_region(
        points=points,
        region=region,
        blocked_mask=blocked_mask,
        margin=margin,
    )

    legs = []
    order = list(range(4))
    if close_loop:
        pairs = [(order[i], order[(i + 1) % 4]) for i in range(4)]
    else:
        pairs = [(order[i], order[i + 1]) for i in range(3)]

    for a, b in pairs:
        s = wp_vids[a]
        g = wp_vids[b]

        out = shortest_path_between_vertices(
            cache=cache,
            cost_e=cost_e,
            theta_e_deg=theta_e_deg,
            start_vid=s,
            goal_vid=g,
            th1=th1, lam1=lam1,
            th2=th2, lam2=lam2,
            use_start_heading=False,
            termination="goal_best",
        )

        legs.append({
            "from_wp_idx": a,
            "to_wp_idx": b,
            "start_vid": s,
            "goal_vid": g,
            **out
        })

    total_cost = 0.0
    full_node_path = []
    full_edge_path = []

    for i, leg in enumerate(legs):
        total_cost += float(leg["best_goal"])
        npth = leg["node_path"]
        epth = leg["edge_path"]

        if len(npth) == 0:
            continue

        if i == 0:
            full_node_path.extend(npth)
        else:
            full_node_path.extend(npth[1:])

        full_edge_path.extend(epth)

    return {
        "region": region,
        "waypoint_xy": wp_xy,
        "waypoint_vids": wp_vids,
        "legs": legs,
        "total_cost": total_cost,
        "full_node_path": full_node_path,
        "full_edge_path": full_edge_path,
    }

# ---------------------------------------------------
# 7) 8개 영역 전체 순찰 계획
# ---------------------------------------------------
def build_all_8_patrols(
    points: np.ndarray,
    cache: EdgeCache,
    cost_e: np.ndarray,
    theta_e_deg: np.ndarray,
    blocked_mask: np.ndarray,
    *,
    x_s: float,
    x_f: float,
    y_s: float,
    y_f: float,
    margin: float = 180.0,
    close_loop: bool = True,
    th1: float = 40.0,
    lam1: float = 10.0,
    th2: float = 50.0,
    lam2: float = np.inf,
) -> List[Dict]:
    regions = make_8_regions(x_s, x_f, y_s, y_f)

    patrols = []
    for region in regions:
        patrol = build_patrol_loop_for_region(
            points=points,
            cache=cache,
            cost_e=cost_e,
            theta_e_deg=theta_e_deg,
            region=region,
            blocked_mask=blocked_mask,
            margin=margin,
            close_loop=close_loop,
            th1=th1, lam1=lam1,
            th2=th2, lam2=lam2,
        )
        patrols.append(patrol)

    return patrols

In [16]:
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, List, Tuple, Optional


# ---------------------------------------------------
# 1) 시작 waypoint를 반영해 순찰 leg 순서를 회전
# ---------------------------------------------------
def rotate_patrol_legs_by_start_wp(patrol: Dict, start_wp_idx: int) -> List[Dict]:
    """
    patrol["legs"]는 기본적으로
      0->1, 1->2, 2->3, 3->0
    순서라고 가정.
    start_wp_idx에서 출발하도록 legs를 회전한다.
    """
    legs = patrol["legs"]
    if len(legs) != 4:
        raise ValueError("Expected exactly 4 patrol legs.")
    s = int(start_wp_idx) % 4
    return legs[s:] + legs[:s]


# ---------------------------------------------------
# 2) 회전된 순찰 경로를 node/edge 단위로 펼치기
# ---------------------------------------------------
def build_rotated_patrol_path(
    patrol: Dict,
    start_wp_idx: int,
) -> Tuple[List[int], List[int]]:
    """
    반환:
      full_node_path: 시작 waypoint 기준으로 회전된 전체 node path
      full_edge_path: 대응 edge path
    """
    legs_rot = rotate_patrol_legs_by_start_wp(patrol, start_wp_idx)

    full_node_path: List[int] = []
    full_edge_path: List[int] = []

    for i, leg in enumerate(legs_rot):
        npth = leg["node_path"]
        epth = leg["edge_path"]

        if len(npth) == 0:
            continue

        if i == 0:
            full_node_path.extend(npth)
        else:
            full_node_path.extend(npth[1:])  # 중복 시작 노드 제거

        full_edge_path.extend(epth)

    return full_node_path, full_edge_path


# ---------------------------------------------------
# 3) 한 USV의 node별 최초 도착 시간 계산
# ---------------------------------------------------
def compute_arrival_times_on_path(
    num_points: int,
    node_path: List[int],
    edge_path: List[int],
    cost_e: np.ndarray,
    *,
    num_laps: int = 1,
) -> np.ndarray:
    """
    순찰 경로를 따라 각 node의 최초 도착 시간을 계산.
    반환:
      arrival_times: (N,)  방문 안한 점은 np.inf
    """
    arr = np.full(num_points, np.inf, dtype=float)

    if len(node_path) == 0:
        return arr

    if len(edge_path) != len(node_path) - 1:
        raise ValueError("edge_path length must be len(node_path)-1")

    t = 0.0
    loop_nodes = node_path
    loop_edges = edge_path

    # 첫 시작점 도착 시간 = 0
    arr[loop_nodes[0]] = 0.0

    for lap in range(num_laps):
        for i, e in enumerate(loop_edges):
            u = loop_nodes[i + 1]
            t += float(cost_e[e])
            if t < arr[u]:
                arr[u] = t

    return arr


# ---------------------------------------------------
# 4) region 내부 점 마스크
# ---------------------------------------------------
def region_point_mask(points: np.ndarray, region) -> np.ndarray:
    return (
        (points[:, 0] >= region.x0) & (points[:, 0] <= region.x1) &
        (points[:, 1] >= region.y0) & (points[:, 1] <= region.y1)
    )


# ---------------------------------------------------
# 5) 8개 USV 전체에 대해 시작 waypoint 무작위 선택 + arrival time 계산
# ---------------------------------------------------
def simulate_multi_usv_patrol_arrival(
    points: np.ndarray,
    patrols: List[Dict],
    cost_e: np.ndarray,
    *,
    seed: Optional[int] = 0,
    num_laps: int = 1,
    assign_only_inside_region: bool = True,
) -> Dict:
    """
    각 patrol(region)마다 4개 waypoint 중 하나를 무작위 시작점으로 정하고
    해당 순찰 루프를 따라 각 격자점의 최초 도착 시간을 계산한다.

    반환:
      {
        "start_wp_indices": List[int],
        "usv_node_paths": List[List[int]],
        "usv_edge_paths": List[List[int]],
        "usv_arrival_times": List[np.ndarray],   # each (N,)
        "global_arrival_times": np.ndarray,      # (N,) min over USVs
      }
    """
    rng = np.random.default_rng(seed)

    start_wp_indices: List[int] = []
    usv_node_paths: List[List[int]] = []
    usv_edge_paths: List[List[int]] = []
    usv_arrival_times: List[np.ndarray] = []

    N = len(points)
    global_arrival_times = np.full(N, np.inf, dtype=float)

    for patrol in patrols:
        start_wp = int(rng.integers(0, 4))
        start_wp_indices.append(start_wp)

        node_path, edge_path = build_rotated_patrol_path(patrol, start_wp)
        usv_node_paths.append(node_path)
        usv_edge_paths.append(edge_path)

        arr = compute_arrival_times_on_path(
            num_points=N,
            node_path=node_path,
            edge_path=edge_path,
            cost_e=cost_e,
            num_laps=num_laps,
        )

        if assign_only_inside_region:
            mask = region_point_mask(points, patrol["region"])
            arr = np.where(mask, arr, np.inf)

        usv_arrival_times.append(arr)
        global_arrival_times = np.minimum(global_arrival_times, arr)

    return {
        "start_wp_indices": start_wp_indices,
        "usv_node_paths": usv_node_paths,
        "usv_edge_paths": usv_edge_paths,
        "usv_arrival_times": usv_arrival_times,
        "global_arrival_times": global_arrival_times,
    }


# ---------------------------------------------------
# 6) 전체 arrival time 시각화
# ---------------------------------------------------
def plot_global_arrival_times(
    points: np.ndarray,
    global_arrival_times: np.ndarray,
    patrols: Optional[List[Dict]] = None,
    sim_result: Optional[Dict] = None,
    *,
    figsize: Tuple[float, float] = (14, 7),
    s: float = 14,
    cmap: str = "viridis",
    title: str = "Earliest arrival time over grid points",
    show_region_boxes: bool = True,
    show_waypoints: bool = True,
    show_start_waypoints: bool = True,
):
    plt.figure(figsize=figsize)

    finite_mask = np.isfinite(global_arrival_times)
    sc = plt.scatter(
        points[finite_mask, 0],
        points[finite_mask, 1],
        c=global_arrival_times[finite_mask],
        s=s,
        cmap=cmap,
    )
    plt.colorbar(sc, label="Arrival time")

    # 도달 못한 점
    inf_mask = ~finite_mask
    if np.any(inf_mask):
        plt.scatter(
            points[inf_mask, 0],
            points[inf_mask, 1],
            s=max(4, s * 0.4),
            c="lightgray",
            alpha=0.6,
            label="Unvisited / unreachable",
        )

    if patrols is not None and show_region_boxes:
        for patrol in patrols:
            rg = patrol["region"]
            xs = [rg.x0, rg.x1, rg.x1, rg.x0, rg.x0]
            ys = [rg.y0, rg.y0, rg.y1, rg.y1, rg.y0]
            plt.plot(xs, ys, "k--", linewidth=0.8, alpha=0.6)
            plt.text(
                0.5 * (rg.x0 + rg.x1),
                0.5 * (rg.y0 + rg.y1),
                f"R{rg.region_id}",
                ha="center",
                va="center",
                fontsize=10,
                alpha=0.8,
            )

    if patrols is not None and show_waypoints:
        for i, patrol in enumerate(patrols):
            wp = patrol["waypoint_xy"]
            plt.scatter(wp[:, 0], wp[:, 1], marker="s", s=70, edgecolors="k")
            for j in range(4):
                plt.text(wp[j, 0] + 20, wp[j, 1] + 20, f"{i}:{j}", fontsize=8)

    if patrols is not None and sim_result is not None and show_start_waypoints:
        starts = sim_result["start_wp_indices"]
        for i, patrol in enumerate(patrols):
            sidx = starts[i]
            wp = patrol["waypoint_xy"][sidx]
            plt.scatter([wp[0]], [wp[1]], marker="*", s=220, edgecolors="k")

    plt.gca().set_aspect("equal", adjustable="box")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(title)
    plt.tight_layout()
    plt.show()


# ---------------------------------------------------
# 7) 각 USV 경로도 같이 시각화
# ---------------------------------------------------
def plot_usv_patrol_paths_with_arrival(
    points: np.ndarray,
    patrols: List[Dict],
    sim_result: Dict,
    *,
    global_arrival_times: Optional[np.ndarray] = None,
    figsize: Tuple[float, float] = (15, 8),
    s_grid: float = 10,
    s_path: float = 1.5,
    cmap: str = "plasma",
    title: str = "USV patrol routes and earliest arrival times",
):
    plt.figure(figsize=figsize)

    if global_arrival_times is not None:
        finite_mask = np.isfinite(global_arrival_times)
        sc = plt.scatter(
            points[finite_mask, 0],
            points[finite_mask, 1],
            c=global_arrival_times[finite_mask],
            s=s_grid,
            cmap=cmap,
            alpha=0.85,
        )
        plt.colorbar(sc, label="Arrival time")
    else:
        plt.scatter(points[:, 0], points[:, 1], s=s_grid, c="lightgray", alpha=0.5)

    for i, patrol in enumerate(patrols):
        region = patrol["region"]
        xs = [region.x0, region.x1, region.x1, region.x0, region.x0]
        ys = [region.y0, region.y0, region.y1, region.y1, region.y0]
        plt.plot(xs, ys, "k--", linewidth=0.8, alpha=0.5)

        node_path = sim_result["usv_node_paths"][i]
        if len(node_path) > 0:
            xy = points[np.array(node_path)]
            plt.plot(xy[:, 0], xy[:, 1], linewidth=s_path, alpha=0.95)

        wp = patrol["waypoint_xy"]
        plt.scatter(wp[:, 0], wp[:, 1], marker="s", s=70, edgecolors="k")

        sidx = sim_result["start_wp_indices"][i]
        swp = wp[sidx]
        plt.scatter([swp[0]], [swp[1]], marker="*", s=220, edgecolors="k")

        plt.text(
            0.5 * (region.x0 + region.x1),
            0.5 * (region.y0 + region.y1),
            f"USV {i}",
            ha="center",
            va="center",
            fontsize=10,
            alpha=0.9,
        )

    plt.gca().set_aspect("equal", adjustable="box")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [17]:
from dataclasses import dataclass
from typing import List, Tuple, Optional
import numpy as np
import matplotlib.pyplot as plt


# ---------------------------------------------------
# 1) 원형 장애물 정의
# ---------------------------------------------------
@dataclass
class CircleObstacle:
    x: float
    y: float
    r: float


# ---------------------------------------------------
# 2) 무작위 원형 장애물 생성
# ---------------------------------------------------
def make_random_circle_obstacles(
    *,
    x_s: float,
    x_f: float,
    y_s: float,
    y_f: float,
    num_obstacles: int = 5,
    radius_range: Tuple[float, float] = (120.0, 240.0),
    border_margin: float = 120.0,
    allow_overlap: bool = False,
    obstacle_clearance: float = 0.0,
    seed: Optional[int] = None,
    max_tries_per_obstacle: int = 3000,
) -> List[CircleObstacle]:
    """
    무작위 원형 장애물 생성

    Args:
      num_obstacles: 장애물 개수
      radius_range: (min_radius, max_radius)
      border_margin: 장애물 중심이 경계에서 최소 이만큼 떨어지도록 강제
      allow_overlap: 장애물끼리 겹침 허용 여부
      obstacle_clearance: 장애물끼리 추가 이격 거리
    """
    rng = np.random.default_rng(seed)

    rmin, rmax = radius_range
    if rmin <= 0 or rmax <= 0 or rmin > rmax:
        raise ValueError("radius_range must satisfy 0 < rmin <= rmax")

    obstacles: List[CircleObstacle] = []

    for _ in range(num_obstacles):
        placed = False

        for _try in range(max_tries_per_obstacle):
            rr = float(rng.uniform(rmin, rmax))

            xmin = x_s + border_margin + rr
            xmax = x_f - border_margin - rr
            ymin = y_s + border_margin + rr
            ymax = y_f - border_margin - rr

            if xmin >= xmax or ymin >= ymax:
                raise ValueError("border_margin / radius_range too large for the domain.")

            cx = float(rng.uniform(xmin, xmax))
            cy = float(rng.uniform(ymin, ymax))

            if not allow_overlap:
                ok = True
                for obs in obstacles:
                    d2 = (cx - obs.x) ** 2 + (cy - obs.y) ** 2
                    min_d = rr + obs.r + obstacle_clearance
                    if d2 < min_d ** 2:
                        ok = False
                        break
                if not ok:
                    continue

            obstacles.append(CircleObstacle(cx, cy, rr))
            placed = True
            break

        if not placed:
            raise RuntimeError(
                f"Failed to place obstacle after {max_tries_per_obstacle} tries. "
                f"Try fewer obstacles, smaller radius_range, or allow_overlap=True."
            )

    return obstacles


# ---------------------------------------------------
# 3) 각 vertex가 장애물 내부인지 판정
# ---------------------------------------------------
def build_blocked_node_mask_from_circles(
    points: np.ndarray,
    obstacles: List[CircleObstacle],
    *,
    inclusive: bool = True,
) -> np.ndarray:
    """
    Returns:
      blocked: (N,) bool
    """
    N = len(points)
    blocked = np.zeros(N, dtype=bool)

    if len(obstacles) == 0:
        return blocked

    px = points[:, 0]
    py = points[:, 1]

    for obs in obstacles:
        d2 = (px - obs.x) ** 2 + (py - obs.y) ** 2
        if inclusive:
            blocked |= (d2 <= obs.r ** 2)
        else:
            blocked |= (d2 < obs.r ** 2)

    return blocked


# ---------------------------------------------------
# 4) blocked node를 adjacency에서 제거
# ---------------------------------------------------
def prune_adjacency_by_blocked_nodes(
    adj: np.ndarray,
    blocked: np.ndarray,
    *,
    missing: int = -1,
) -> np.ndarray:
    """
    blocked node로 들어가거나, blocked node에서 나가는 연결 제거
    cache의 대칭성 가정이 깨지지 않도록 노드 자체를 그래프에서 제거하는 방식
    """
    adj2 = np.array(adj, copy=True)
    N, K = adj2.shape

    if blocked.shape[0] != N:
        raise ValueError("blocked mask length must match number of nodes")

    # 1) blocked node에서 나가는 edge 제거
    adj2[blocked, :] = missing

    # 2) blocked node로 들어가는 edge 제거
    for v in range(N):
        if blocked[v]:
            continue
        row = adj2[v]
        valid = (row != missing)
        nbrs = row[valid]
        if len(nbrs) == 0:
            continue
        bad = blocked[nbrs]
        row_idx = np.where(valid)[0][bad]
        adj2[v, row_idx] = missing

    return adj2


# ---------------------------------------------------
# 5) 장애물 적용 전체 래퍼
# ---------------------------------------------------
def apply_circle_obstacles_to_graph(
    points: np.ndarray,
    adj: np.ndarray,
    obstacles: List[CircleObstacle],
    *,
    missing: int = -1,
    inclusive: bool = True,
):
    """
    Returns:
      blocked_mask: (N,) bool
      adj_pruned: (N,K)
    """
    blocked_mask = build_blocked_node_mask_from_circles(
        points,
        obstacles,
        inclusive=inclusive,
    )
    adj_pruned = prune_adjacency_by_blocked_nodes(
        adj,
        blocked_mask,
        missing=missing,
    )
    return blocked_mask, adj_pruned


# ---------------------------------------------------
# 6) 시각화: 장애물 + blocked node
# ---------------------------------------------------
def plot_obstacles_and_blocked_nodes(
    points: np.ndarray,
    obstacles: List[CircleObstacle],
    blocked_mask: np.ndarray,
    *,
    figsize: Tuple[float, float] = (14, 7),
    s_all: float = 8,
    s_blocked: float = 18,
    title: str = "Random circular obstacles and blocked nodes",
):
    fig, ax = plt.subplots(figsize=figsize)

    free_mask = ~blocked_mask
    ax.scatter(points[free_mask, 0], points[free_mask, 1], s=s_all, alpha=0.5, label="Free nodes")
    ax.scatter(points[blocked_mask, 0], points[blocked_mask, 1], s=s_blocked, alpha=0.9, label="Blocked nodes")

    for i, obs in enumerate(obstacles):
        circ = plt.Circle((obs.x, obs.y), obs.r, fill=False, linewidth=2)
        ax.add_patch(circ)
        ax.text(obs.x, obs.y, f"O{i}", ha="center", va="center")

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [18]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

# ---------------------------------------------------
# optional ML backends
# ---------------------------------------------------
try:
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge
    from sklearn.ensemble import RandomForestRegressor
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False


# ---------------------------------------------------
# dataclasses
# ---------------------------------------------------
@dataclass
class TravelSample:
    start_idx: int
    goal_idx: int
    start_heading_deg: float
    travel_time: float
    feature_dict: Dict[str, float]


@dataclass
class TrainResult:
    model_name: str
    target_mode: str
    model: Any
    feature_names: List[str]

    X_train: np.ndarray
    X_test: np.ndarray
    y_train: np.ndarray
    y_test: np.ndarray

    y_pred_train: np.ndarray
    y_pred_test: np.ndarray

    y_true_train_time: np.ndarray
    y_true_test_time: np.ndarray
    y_pred_train_time: np.ndarray
    y_pred_test_time: np.ndarray


# ---------------------------------------------------
# geometry helpers
# ---------------------------------------------------
def unit_vector(vec: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(vec)
    if n < eps:
        return np.zeros_like(vec, dtype=float)
    return vec / n


def point_to_segment_distance_batch(points: np.ndarray, a: np.ndarray, b: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    returns:
      t_proj      : projection ratio on segment line
      perp_dist   : perpendicular distance to infinite line projection
      seg_len     : segment length
    """
    ab = b - a
    L = np.linalg.norm(ab)
    if L < 1e-12:
        t_proj = np.zeros(len(points), dtype=float)
        perp_dist = np.linalg.norm(points - a[None, :], axis=1)
        return t_proj, perp_dist, 0.0

    ab_u = ab / L
    ap = points - a[None, :]
    proj_len = ap @ ab_u
    t_proj = proj_len / L
    proj = a[None, :] + np.outer(proj_len, ab_u)
    perp_dist = np.linalg.norm(points - proj, axis=1)
    return t_proj, perp_dist, float(L)


def random_heading_deg(rng: np.random.Generator) -> float:
    return float(rng.uniform(0.0, 360.0))


def angle_from_vec_y_clockwise_deg(vec: np.ndarray) -> float:
    dx, dy = float(vec[0]), float(vec[1])
    ang = np.degrees(np.arctan2(dx, dy))
    return float((ang + 360.0) % 360.0)


def ang_diff_abs_deg(a_deg: float, b_deg: float) -> float:
    return abs(wrap180_deg(float(a_deg) - float(b_deg)))


# ---------------------------------------------------
# feature extraction
# ---------------------------------------------------


def extract_fixed_features_with_outer_gap(
    *,
    points: np.ndarray,
    blocked_mask: np.ndarray,
    total_ux: np.ndarray,
    total_uy: np.ndarray,
    start_idx: int,
    goal_idx: int,
    start_heading_deg: float,
    inner_width: float = 90.0,
    outer_width: float = 180.0,
) -> Dict[str, float]:
    """
    feature:
      1) inner perpendicular current mean
      2) inner perpendicular current var
      3) inner parallel current mean
      4) inner parallel current var
      5) heading-goal difference
      6) distance
      7) blocked ratio (inner corridor 기준)
      8) parallel outer-inner mean gap
      9) perp outer-inner mean gap
    """

    p0 = points[start_idx]
    p1 = points[goal_idx]
    goal_vec = p1 - p0
    dist = float(np.linalg.norm(goal_vec))

    if dist < 1e-12:
        return {
            "perp_current_mean": 0.0,
            "perp_current_var": 0.0,
            "parallel_current_mean": 0.0,
            "parallel_current_var": 0.0,
            "heading_goal_diff":0.0,
            "distance": 0.0,
            "blocked_ratio": 0.0,
            "parallel_outer_inner_gap_mean_abs": 0.0,
            "perp_outer_inner_gap_mean_abs": 0.0,
        }

    dir_u = goal_vec / dist

    # 각 점의 직선 기준 signed perpendicular distance 계산
    ap = points - p0[None, :]
    t_along = ap @ dir_u  # 직선 방향 투영 길이

    # signed perpendicular:
    # left/right 구분용
    # 2D cross product scalar = ap_x * dir_y - ap_y * dir_x
    signed_perp = ap[:, 0] * dir_u[1] - ap[:, 1] * dir_u[0]
    abs_perp = np.abs(signed_perp)

    # 선분 내부만 사용
    seg_mask = (t_along >= 0.0) & (t_along <= dist)

    inner_mask = seg_mask & (abs_perp <= inner_width)
    left_outer_mask = seg_mask & (signed_perp > inner_width) & (signed_perp <= outer_width)
    right_outer_mask = seg_mask & (signed_perp < -inner_width) & (signed_perp >= -outer_width)

    # 해류를 goal 방향 기준으로 분해
    parallel = total_ux * dir_u[0] + total_uy * dir_u[1]
    perpendicular = -total_ux * dir_u[1] + total_uy * dir_u[0]

    # blocked ratio는 inner 기준
    inner_total = int(np.sum(inner_mask))
    inner_blocked = int(np.sum(blocked_mask[inner_mask]))
    blocked_ratio = float(inner_blocked / max(inner_total, 1))

    # 통계는 free node 기준으로
    inner_stat_mask = inner_mask & (~blocked_mask)
    left_outer_stat_mask = left_outer_mask & (~blocked_mask)
    right_outer_stat_mask = right_outer_mask & (~blocked_mask)

    def safe_mean_var(arr: np.ndarray, mask: np.ndarray):
        vals = arr[mask]
        if vals.size == 0:
            return 0.0, 0.0
        return float(np.mean(vals)), float(np.var(vals))

    # inner stats
    par_in_mean, par_in_var = safe_mean_var(parallel, inner_stat_mask)
    perp_in_mean, perp_in_var = safe_mean_var(perpendicular, inner_stat_mask)

    # outer stats
    par_left_mean, _ = safe_mean_var(parallel, left_outer_stat_mask)
    par_right_mean, _ = safe_mean_var(parallel, right_outer_stat_mask)

    perp_left_mean, _ = safe_mean_var(perpendicular, left_outer_stat_mask)
    perp_right_mean, _ = safe_mean_var(perpendicular, right_outer_stat_mask)

    # 너가 원하는 gap feature
    parallel_outer_inner_gap_mean_abs = 0.5 * (
        abs(par_left_mean - par_in_mean) + abs(par_right_mean - par_in_mean)
    )

    perp_outer_inner_gap_mean_abs = 0.5 * (
        abs(perp_left_mean - perp_in_mean) + abs(perp_right_mean - perp_in_mean)
    )

    goal_heading_deg = angle_from_vec_y_clockwise_deg(goal_vec)
    heading_goal_diff_deg = abs(wrap180_deg(start_heading_deg - goal_heading_deg))
    return {
        "perp_current_mean": perp_in_mean,
        "perp_current_var": perp_in_var,
        "parallel_current_mean": par_in_mean,
        "parallel_current_var": par_in_var,
        "heading_goal_diff":heading_goal_diff_deg,
        "distance": dist,
        "blocked_ratio": blocked_ratio,
        "parallel_outer_inner_gap_mean_abs": float(parallel_outer_inner_gap_mean_abs),
        "perp_outer_inner_gap_mean_abs": float(perp_outer_inner_gap_mean_abs),
    }
# ---------------------------------------------------
# random feasible pair
# ---------------------------------------------------
def sample_start_goal(
    *,
    points: np.ndarray,
    blocked_mask: np.ndarray,
    adj_pruned: np.ndarray,
    rng: np.random.Generator,
    min_dist: float = 300.0,
    missing: int = -1,
    max_tries: int = 5000,
) -> Tuple[int, int]:
    deg = np.sum(adj_pruned != missing, axis=1)
    candidates = np.where((~blocked_mask) & (deg > 0))[0]

    if len(candidates) < 2:
        raise RuntimeError("Not enough feasible candidate nodes.")

    for _ in range(max_tries):
        s, g = rng.choice(candidates, size=2, replace=False)
        if np.linalg.norm(points[g] - points[s]) >= min_dist:
            return int(s), int(g)

    raise RuntimeError("Failed to sample feasible start-goal pair.")


# ---------------------------------------------------
# one random scenario -> one labeled sample
# ---------------------------------------------------

def generate_one_sample(
    *,
    points: np.ndarray,
    adj_base: np.ndarray,
    domain: Dict[str, float],
    usv_speed: float,
    rng: np.random.Generator,

    # obstacle params
    num_obstacles: int = 0,
    radius_range: Tuple[float, float] = (120.0, 200.0),
    border_margin: float = 50.0,
    obstacle_clearance: float = 200.0,

    # path / feature params
    min_start_goal_dist: float = 300.0,
    corridor_width: float = 90.0,

    # turn constraint
    th1: float = 40.0,
    lam1: float = 10.0,
    th2: float = 50.0,
    lam2: float = np.inf,
) -> Optional[TravelSample]:
    # 1) random current field
    vortices, uniform = make_random_vortices(
        Lx=domain["x_f"] - domain["x_s"],
        Ly=domain["y_f"] - domain["y_s"],
        seed=int(rng.integers(0, 10**9)),
    )

    cur = composite_current(points, vortices, uniform=uniform, return_per_vortex=False)
    total_ux = cur["total_ux"]
    total_uy = cur["total_uy"]
    current_speed = cur["speed"]
    current_angle_deg = cur["angle_deg"]

    # 2) random obstacles
    obstacles = make_random_circle_obstacles(
        x_s=domain["x_s"], x_f=domain["x_f"],
        y_s=domain["y_s"], y_f=domain["y_f"],
        num_obstacles=num_obstacles,
        radius_range=radius_range,
        border_margin=border_margin,
        allow_overlap=False,
        obstacle_clearance=obstacle_clearance,
        seed=int(rng.integers(0, 10**9)),
    )

    blocked_mask, adj_pruned = apply_circle_obstacles_to_graph(
        points=points,
        adj=adj_base,
        obstacles=obstacles,
        missing=-1,
        inclusive=True,
    )

    # 장애물 때문에 그래프가 너무 망가진 경우 skip
    try:
        cache = build_edge_cache(points.copy(), adj_pruned.copy())
    except Exception:
        return None

    # 3) edge travel time under current
    theta_deg_e, denom_e, cost_e, feasible_e, valid_e = compute_theta_cost_from_cache(
        cache,
        theta_f_deg=current_angle_deg,
        v_f=current_speed,
        V=usv_speed,
    )

    if not np.any(np.isfinite(cost_e)):
        return None

    # 4) random start/goal + heading
    try:
        start_idx, goal_idx = sample_start_goal(
            points=points,
            blocked_mask=blocked_mask,
            adj_pruned=adj_pruned,
            rng=rng,
            min_dist=min_start_goal_dist,
        )
    except RuntimeError:
        return None

    start_heading_deg = random_heading_deg(rng)

    # 5) label with dijkstra
    dist_state, prev_node, prev_k, best_goal, best_goal_k = dijkstra_turn_state_core(
        cache=cache,
        base_cost_e=cost_e,
        theta_e_deg=theta_deg_e,
        start=start_idx,
        goal=goal_idx,
        th1=th1, lam1=lam1,
        th2=th2, lam2=lam2,
        use_start_heading=True,
        start_heading_deg=start_heading_deg,
        termination="goal_best",
    )

    if (not np.isfinite(best_goal)) or (best_goal_k < 0):
        return None

    # 6) fixed features only
    feat = extract_fixed_features_with_outer_gap(
        points=points,
        blocked_mask=blocked_mask,
        total_ux=total_ux,
        total_uy=total_uy,
        start_idx=start_idx,
        goal_idx=goal_idx,
        start_heading_deg=start_heading_deg,
        inner_width=90.0,
        outer_width=180.0
    )

    return TravelSample(
        start_idx=start_idx,
        goal_idx=goal_idx,
        start_heading_deg=start_heading_deg,
        travel_time=float(best_goal),
        feature_dict=feat,
    )


# ---------------------------------------------------
# dataset generation
# ---------------------------------------------------

def generate_dataset(
    *,
    n_samples: int,
    points: np.ndarray,
    adj_base: np.ndarray,
    domain: Dict[str, float],
    usv_speed: float,
    rng_seed: int = 123,

    num_obstacles: int = 0,
    radius_range: Tuple[float, float] = (120.0, 200.0),
    border_margin: float = 50.0,
    obstacle_clearance: float = 200.0,

    min_start_goal_dist: float = 300.0,
    corridor_width: float = 90.0,

    th1: float = 40.0,
    lam1: float = 10.0,
    th2: float = 50.0,
    lam2: float = np.inf,

    max_total_tries: int = 100000,
) -> List[TravelSample]:
    rng = np.random.default_rng(rng_seed)
    samples: List[TravelSample] = []

    tries = 0
    while len(samples) < n_samples and tries < max_total_tries:
        if tries%100==0:
            print(tries)
        tries += 1

        s = generate_one_sample(
            points=points,
            adj_base=adj_base,
            domain=domain,
            usv_speed=usv_speed,
            rng=rng,
            num_obstacles=num_obstacles,
            radius_range=radius_range,
            border_margin=border_margin,
            obstacle_clearance=obstacle_clearance,
            min_start_goal_dist=min_start_goal_dist,
            corridor_width=corridor_width,
            th1=th1,
            lam1=lam1,
            th2=th2,
            lam2=lam2,
        )

        if s is not None:
            samples.append(s)

    return samples
def save_samples_pickle(samples, filepath: str):
    with open(filepath, "wb") as f:
        pickle.dump(samples, f)


def load_samples_pickle(filepath: str):
    with open(filepath, "rb") as f:
        samples = pickle.load(f)
    return samples
# ---------------------------------------------------
# convert samples -> X, y
# ---------------------------------------------------
def samples_to_xy(
    samples: List[TravelSample],
    *,
    points: np.ndarray,
    usv_speed: float,
    target_mode: str = "absolute",   # "absolute" or "residual_percent"
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[str]]:
    if len(samples) == 0:
        raise ValueError("No samples.")

    feature_names = list(samples[0].feature_dict.keys())

    X = []
    y_true = []
    y_base = []

    for s in samples:
        X.append([s.feature_dict[k] for k in feature_names])
        y_true.append(float(s.travel_time))

        p0 = points[s.start_idx]
        p1 = points[s.goal_idx]
        base_time = float(np.linalg.norm(p1 - p0) / usv_speed)
        y_base.append(base_time)

    X = np.asarray(X, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    y_base = np.asarray(y_base, dtype=float)

    if target_mode == "absolute":
        y = y_true.copy()
    elif target_mode == "residual_percent":
        y = 100.0 * (y_true - y_base) / np.maximum(y_base, 1e-12)
    else:
        raise ValueError("target_mode must be 'absolute' or 'residual_percent'.")

    return X, y, y_true, y_base, feature_names


# ---------------------------------------------------
# model builder
# ---------------------------------------------------
def build_regressor(model_name: str, random_state: int = 42, **kwargs):
    model_name = model_name.lower().strip()

    if model_name == "ridge":
        if not SKLEARN_AVAILABLE:
            raise ImportError("scikit-learn is required for ridge.")
        alpha = kwargs.get("alpha", 3.0)
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=alpha))
        ])

    elif model_name == "rf":
        if not SKLEARN_AVAILABLE:
            raise ImportError("scikit-learn is required for random forest.")
        return RandomForestRegressor(
            n_estimators=kwargs.get("n_estimators", 400),
            max_depth=kwargs.get("max_depth", None),
            min_samples_leaf=kwargs.get("min_samples_leaf", 1),
            random_state=random_state,
            n_jobs=-1,
        )

    elif model_name == "xgb":
        if not XGB_AVAILABLE:
            raise ImportError("xgboost is not installed.")
        return XGBRegressor(
            n_estimators=kwargs.get("n_estimators", 500),
            max_depth=kwargs.get("max_depth", 5),
            learning_rate=kwargs.get("learning_rate", 0.05),
            subsample=kwargs.get("subsample", 0.9),
            colsample_bytree=kwargs.get("colsample_bytree", 0.9),
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
        )

    else:
        raise ValueError("model_name must be 'ridge', 'rf', or 'xgb'.")


# ---------------------------------------------------
# metrics
# ---------------------------------------------------
def mape_percent(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-12) -> float:
    return float(np.mean(np.abs(y_pred - y_true) / np.maximum(np.abs(y_true), eps)) * 100.0)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    err = y_pred - y_true
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 1e-12 else 0.0
    mape = mape_percent(y_true, y_pred)
    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE_%": mape,
    }


def recover_time_from_target(y_pred: np.ndarray, y_base: np.ndarray, target_mode: str) -> np.ndarray:
    if target_mode == "absolute":
        return y_pred.copy()
    elif target_mode == "residual_percent":
        return y_base * (1.0 + y_pred / 100.0)
    else:
        raise ValueError("Invalid target_mode")


# ---------------------------------------------------
# training
# ---------------------------------------------------
def train_model(
    *,
    samples: List[TravelSample],
    points: np.ndarray,
    usv_speed: float,
    target_mode: str = "absolute",
    model_name: str = "ridge",
    test_size: float = 0.2,
    split_seed: int = 42,
    **model_kwargs,
) -> TrainResult:
    if not SKLEARN_AVAILABLE:
        raise ImportError("scikit-learn is required for splitting/training.")

    X, y, y_true, y_base, feature_names = samples_to_xy(
        samples,
        points=points,
        usv_speed=usv_speed,
        target_mode=target_mode,
    )

    X_train, X_test, y_train, y_test, y_true_train, y_true_test, y_base_train, y_base_test = train_test_split(
        X, y, y_true, y_base,
        test_size=test_size,
        random_state=split_seed,
    )

    model = build_regressor(model_name, random_state=split_seed, **model_kwargs)
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    y_pred_train_time = recover_time_from_target(y_pred_train, y_base_train, target_mode)
    y_pred_test_time = recover_time_from_target(y_pred_test, y_base_test, target_mode)

    return TrainResult(
        model_name=model_name,
        target_mode=target_mode,
        model=model,
        feature_names=feature_names,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        y_pred_train=y_pred_train,
        y_pred_test=y_pred_test,
        y_true_train_time=y_true_train,
        y_true_test_time=y_true_test,
        y_pred_train_time=y_pred_train_time,
        y_pred_test_time=y_pred_test_time,
    )


# ---------------------------------------------------
# result summary
# ---------------------------------------------------
def summarize_result(result: TrainResult) -> pd.DataFrame:
    train_target = regression_metrics(result.y_train, result.y_pred_train)
    test_target = regression_metrics(result.y_test, result.y_pred_test)
    train_time = regression_metrics(result.y_true_train_time, result.y_pred_train_time)
    test_time = regression_metrics(result.y_true_test_time, result.y_pred_test_time)

    rows = [
        {"split": "train_target", **train_target},
        {"split": "test_target", **test_target},
        {"split": "train_time", **train_time},
        {"split": "test_time", **test_time},
    ]
    return pd.DataFrame(rows)


def compare_results(results: List[TrainResult]) -> pd.DataFrame:
    rows = []
    for r in results:
        rows.append({
            "model": r.model_name,
            "target_mode": r.target_mode,
            "test_time_MAPE_%": mape_percent(r.y_true_test_time, r.y_pred_test_time),
            "test_time_RMSE": float(np.sqrt(np.mean((r.y_pred_test_time - r.y_true_test_time) ** 2))),
        })
    return pd.DataFrame(rows).sort_values("test_time_MAPE_%").reset_index(drop=True)


def print_feature_importance(result: TrainResult, top_k: int = 10):
    name = result.model_name.lower()

    if name == "ridge":
        coef = result.model.named_steps["model"].coef_
        idx = np.argsort(np.abs(coef))[::-1]
        print("\n[feature weights]")
        for i in idx[:top_k]:
            print(f"{result.feature_names[i]:25s}: {coef[i]: .6f}")

    elif name in ("rf", "xgb"):
        imp = result.model.feature_importances_
        idx = np.argsort(imp)[::-1]
        print("\n[feature importances]")
        for i in idx[:top_k]:
            print(f"{result.feature_names[i]:25s}: {imp[i]: .6f}")

    else:
        print("importance not supported")


def baseline_mape_from_result(
    result: TrainResult,
    *,
    samples: List[TravelSample],
    points: np.ndarray,
    usv_speed: float,
    test_size: float = 0.2,
    split_seed: int = 42,
) -> float:
    """
    FitResult를 수정하지 않고 baseline test MAPE를 따로 계산
    """
    X, y_abs, y_true, y_base, feature_names = samples_to_xy(
        samples,
        points=points,
        usv_speed=usv_speed,
        target_mode="absolute",
    )

    _, _, _, _, y_true_train, y_true_test, y_base_train, y_base_test = train_test_split(
        X, y_abs, y_true, y_base,
        test_size=test_size,
        random_state=split_seed,
    )
    return mape_percent(y_true_test, y_base_test),float(np.sqrt(np.mean((y_base_test - y_true_test) ** 2)))

In [31]:
# ---------------------------------------------------
# A. 고정 mesh / base adjacency
# ---------------------------------------------------
x_s, x_f = 0.0, 3600.0
y_s, y_f = 0.0, 3600.0
x_c, y_c = 0.0, 0.0
r = 60.0
mod = "Hexa"
neigh_mode = "Extra_extended_edges"

points, idx_map = mesh(x_s, x_f, y_s, y_f, x_c, y_c, r, mod=mod)
adj_base = build_vertex_adjacency(idx_map.copy(), len(points), mod=mod, mode=neigh_mode)

domain = {
    "x_s": x_s,
    "x_f": x_f,
    "y_s": y_s,
    "y_f": y_f,
}
# ---------------------------------------------------
# B. dataset generation
# ---------------------------------------------------
'''
samples = generate_dataset(
    n_samples=5000,
    points=points,
    adj_base=adj_base,
    domain=domain,
    usv_speed=2.5,
    rng_seed=123,

    num_obstacles=0,
    radius_range=(120.0, 200.0),
    border_margin=50.0,
    obstacle_clearance=200.0,

    min_start_goal_dist=200.0,
    corridor_width=90.0,

    th1=40.0,
    lam1=0.0,
    th2=60.0,
    lam2=np.inf,
)
'''
import pickle
#save_samples_pickle(samples, "5000_samples_60_final.pkl")

#print("saved samples:", len(samples))
samples=load_samples_pickle("5000_samples_60_final.pkl")

print("num samples =", len(samples))
print("feature names =", list(samples[0].feature_dict.keys()))
result_abs_ridge = train_model(
    samples=samples,
    points=points,
    usv_speed=2.5,
    target_mode="absolute",
    model_name="ridge",
    test_size=0.2,
    split_seed=42,
    alpha=3.0,
)

result_abs_rf = train_model(
    samples=samples,
    points=points,
    usv_speed=2.5,
    target_mode="absolute",
    model_name="rf",
    test_size=0.2,
    split_seed=42,
    n_estimators=400,
    max_depth=14,
    min_samples_leaf=2,
)

result_abs_xgb = train_model(
    samples=samples,
    points=points,
    usv_speed=2.5,
    target_mode="absolute",
    model_name="xgb",
    test_size=0.2,
    split_seed=42,
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
)
result_res_ridge = train_model(
    samples=samples,
    points=points,
    usv_speed=2.5,
    target_mode="residual_percent",
    model_name="ridge",
    test_size=0.2,
    split_seed=42,
    alpha=3.0,
)

result_res_rf = train_model(
    samples=samples,
    points=points,
    usv_speed=2.5,
    target_mode="residual_percent",
    model_name="rf",
    test_size=0.2,
    split_seed=42,
    n_estimators=400,
    max_depth=14,
    min_samples_leaf=2,
)

result_res_xgb = train_model(
    samples=samples,
    points=points,
    usv_speed=2.5,
    target_mode="residual_percent",
    model_name="xgb",
    test_size=0.2,
    split_seed=42,
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
)
df_compare = compare_results([
    result_abs_ridge,
    result_abs_rf,
    result_abs_xgb,
    result_res_ridge,
    result_res_rf,
    result_res_xgb,
])

print(df_compare.to_string(index=False))
baseline_test_mape = baseline_mape_from_result(
    result_abs_ridge,
    samples=samples,
    points=points,
    usv_speed=2.5,
    test_size=0.2,
    split_seed=42,
)

print("baseline test MAPE (%) =", baseline_test_mape)

num samples = 5000
feature names = ['perp_current_mean', 'perp_current_var', 'parallel_current_mean', 'parallel_current_var', 'heading_goal_diff', 'distance', 'blocked_ratio', 'parallel_outer_inner_gap_mean_abs', 'perp_outer_inner_gap_mean_abs']
model      target_mode  test_time_MAPE_%  test_time_RMSE
  xgb residual_percent          3.373435       45.459437
   rf residual_percent          3.887531       51.866781
  xgb         absolute          3.947033       49.844278
   rf         absolute          4.833451       58.102371
ridge residual_percent          8.394567       86.515653
ridge         absolute         10.839745       83.302090
baseline test MAPE (%) = (18.092060582445733, 197.85655698294676)


In [7]:
import joblib
x_s, x_f = 0.0, 3600.0
y_s, y_f = 0.0, 3600.0
x_c, y_c = 0.0, 0.0
r = 60.0
mod = "Hexa"
neigh_mode = "Extra_extended_edges"

points, idx_map = mesh(x_s, x_f, y_s, y_f, x_c, y_c, r, mod=mod)
adj_base = build_vertex_adjacency(idx_map.copy(), len(points), mod=mod, mode=neigh_mode)

domain = {
    "x_s": x_s,
    "x_f": x_f,
    "y_s": y_s,
    "y_f": y_f,
}

In [9]:
from dataclasses import dataclass
from typing import Optional, List, Dict, Any, Tuple
import numpy as np
import pandas as pd
import time


@dataclass
class RandomBenchmarkCase:
    case_id: int

    # graph/scenario
    blocked_mask: np.ndarray
    adj_pruned: np.ndarray
    cache: EdgeCache

    total_ux: np.ndarray
    total_uy: np.ndarray
    current_speed: np.ndarray
    current_angle_deg: np.ndarray

    cost_e: np.ndarray
    theta_e_deg: np.ndarray

    start_vid: int
    goal_vid: int
    start_heading_deg: float

    # baseline label
    dijkstra_best_time: float


def generate_one_benchmark_case(
    *,
    case_id: int,
    points: np.ndarray,
    adj_base: np.ndarray,
    domain: Dict[str, float],
    usv_speed: float,
    rng: np.random.Generator,

    # obstacle params
    num_obstacles: int = 0,
    radius_range: Tuple[float, float] = (120.0, 200.0),
    border_margin: float = 50.0,
    obstacle_clearance: float = 200.0,

    # start-goal params
    min_start_goal_dist: float = 300.0,

    # turn constraint
    th1: float = 40.0,
    lam1: float = 10.0,
    th2: float = 50.0,
    lam2: float = np.inf,
) -> Optional[RandomBenchmarkCase]:
    # 1) random current field
    vortices, uniform = make_random_vortices(
        Lx=domain["x_f"] - domain["x_s"],
        Ly=domain["y_f"] - domain["y_s"],
        seed=int(rng.integers(0, 10**9)),
    )

    cur = composite_current(points, vortices, uniform=uniform, return_per_vortex=False)
    total_ux = cur["total_ux"]
    total_uy = cur["total_uy"]
    current_speed = cur["speed"]
    current_angle_deg = cur["angle_deg"]

    # 2) random obstacles
    obstacles = make_random_circle_obstacles(
        x_s=domain["x_s"], x_f=domain["x_f"],
        y_s=domain["y_s"], y_f=domain["y_f"],
        num_obstacles=num_obstacles,
        radius_range=radius_range,
        border_margin=border_margin,
        allow_overlap=False,
        obstacle_clearance=obstacle_clearance,
        seed=int(rng.integers(0, 10**9)),
    )

    blocked_mask, adj_pruned = apply_circle_obstacles_to_graph(
        points=points,
        adj=adj_base,
        obstacles=obstacles,
        missing=-1,
        inclusive=True,
    )

    # 3) cache
    try:
        cache = build_edge_cache(points.copy(), adj_pruned.copy())
    except Exception:
        return None

    # 4) edge travel time
    theta_deg_e, denom_e, cost_e, feasible_e, valid_e = compute_theta_cost_from_cache(
        cache,
        theta_f_deg=current_angle_deg,
        v_f=current_speed,
        V=usv_speed,
    )

    if not np.any(np.isfinite(cost_e)):
        return None

    # 5) random start-goal
    try:
        start_vid, goal_vid = sample_start_goal(
            points=points,
            blocked_mask=blocked_mask,
            adj_pruned=adj_pruned,
            rng=rng,
            min_dist=min_start_goal_dist,
        )
    except RuntimeError:
        return None

    # 6) random start heading
    start_heading_deg = random_heading_deg(rng)

    # 7) baseline label by Dijkstra
    dist_state, prev_node, prev_k, best_goal, best_goal_k = dijkstra_turn_state_core(
        cache=cache,
        base_cost_e=cost_e,
        theta_e_deg=theta_deg_e,
        start=start_vid,
        goal=goal_vid,
        th1=th1, lam1=lam1,
        th2=th2, lam2=lam2,
        use_start_heading=True,
        start_heading_deg=start_heading_deg,
        termination="goal_best",
    )

    if (not np.isfinite(best_goal)) or (best_goal_k < 0):
        return None

    return RandomBenchmarkCase(
        case_id=case_id,
        blocked_mask=blocked_mask,
        adj_pruned=adj_pruned,
        cache=cache,
        total_ux=total_ux,
        total_uy=total_uy,
        current_speed=current_speed,
        current_angle_deg=current_angle_deg,
        cost_e=cost_e,
        theta_e_deg=theta_deg_e,
        start_vid=int(start_vid),
        goal_vid=int(goal_vid),
        start_heading_deg=float(start_heading_deg),
        dijkstra_best_time=float(best_goal),
    )


def generate_random_benchmark_cases(
    *,
    n_cases: int,
    points: np.ndarray,
    adj_base: np.ndarray,
    domain: Dict[str, float],
    usv_speed: float,
    rng_seed: int = 123,

    num_obstacles: int = 0,
    radius_range: Tuple[float, float] = (120.0, 200.0),
    border_margin: float = 50.0,
    obstacle_clearance: float = 200.0,
    min_start_goal_dist: float = 300.0,

    th1: float = 40.0,
    lam1: float = 10.0,
    th2: float = 50.0,
    lam2: float = np.inf,

    max_total_tries: int = 100000,
    verbose_every: int = 20,
) -> List[RandomBenchmarkCase]:
    rng = np.random.default_rng(rng_seed)
    cases: List[RandomBenchmarkCase] = []

    tries = 0
    next_case_id = 0

    while len(cases) < n_cases and tries < max_total_tries:
        tries += 1

        case = generate_one_benchmark_case(
            case_id=next_case_id,
            points=points,
            adj_base=adj_base,
            domain=domain,
            usv_speed=usv_speed,
            rng=rng,
            num_obstacles=num_obstacles,
            radius_range=radius_range,
            border_margin=border_margin,
            obstacle_clearance=obstacle_clearance,
            min_start_goal_dist=min_start_goal_dist,
            th1=th1,
            lam1=lam1,
            th2=th2,
            lam2=lam2,
        )

        if case is not None:
            cases.append(case)
            next_case_id += 1
            if verbose_every is not None and len(cases) % verbose_every == 0:
                print(f"generated {len(cases)} / {n_cases} cases")

    if len(cases) < n_cases:
        raise RuntimeError(f"Only generated {len(cases)} cases before max_total_tries={max_total_tries}")

    return cases

In [10]:
def predict_direct_model_time_for_case(
    *,
    case: RandomBenchmarkCase,
    points: np.ndarray,
    model,
    feature_names: List[str],
    target_mode: str,
    usv_speed: float,
    inner_width: float = 90.0,
    outer_width: float = 180.0,
) -> float:
    feat = extract_fixed_features_with_outer_gap(
        points=points,
        blocked_mask=case.blocked_mask,
        total_ux=case.total_ux,
        total_uy=case.total_uy,
        start_idx=case.start_vid,
        goal_idx=case.goal_vid,
        start_heading_deg=case.start_heading_deg,
        inner_width=inner_width,
        outer_width=outer_width,
    )

    x = np.array([[feat[name] for name in feature_names]], dtype=float)
    y_pred = np.asarray(model.predict(x), dtype=float).reshape(-1)

    p0 = points[case.start_vid]
    p1 = points[case.goal_vid]
    base_time = float(np.linalg.norm(p1 - p0) / usv_speed)
    y_base = np.array([base_time], dtype=float)

    pred_time = float(recover_time_from_target(y_pred, y_base, target_mode)[0])
    return max(pred_time, 0.0)
def benchmark_four_methods_on_cases(
    *,
    cases: List[RandomBenchmarkCase],
    points: np.ndarray,
    usv_speed: float,

    # method 2
    learned_model=None,
    learned_feature_names=None,
    learned_target_mode: str = "absolute",

    # method 4
    surrogate_model=None,
    surrogate_feature_names=None,
    surrogate_target_mode: str = "absolute",

    inner_width: float = 90.0,
    outer_width: float = 180.0,

    th1: float = 40.0,
    lam1: float = 10.0,
    th2: float = 50.0,
    lam2: float = np.inf,

    use_start_heading: bool = True,
) -> pd.DataFrame:
    rows = []

    for case in cases:
        baseline = float(case.dijkstra_best_time)

        # ---------------------------------------
        # 1) dijkstra baseline
        # ---------------------------------------
        t0 = time.perf_counter()
        res_dij = shortest_path_between_vertices(
            cache=case.cache,
            cost_e=case.cost_e,
            theta_e_deg=case.theta_e_deg,
            start_vid=case.start_vid,
            goal_vid=case.goal_vid,
            th1=th1, lam1=lam1,
            th2=th2, lam2=lam2,
            use_start_heading=use_start_heading,
            start_heading_deg=case.start_heading_deg,
            termination="goal_best",
        )
        t1 = time.perf_counter()

        rows.append({
            "case_id": case.case_id,
            "method": "dijkstra",
            "travel_time": float(res_dij["best_goal"]),
            "runtime_sec": float(t1 - t0),
            "abs_error_vs_dijkstra": 0.0,
            "relative_error_vs_dijkstra_percent": 0.0,
            "path_gap_vs_dijkstra": 0.0,
            "rel_gap_vs_dijkstra_percent": 0.0,
            "start_vid": case.start_vid,
            "goal_vid": case.goal_vid,
            "start_heading_deg": case.start_heading_deg,
        })

        # ---------------------------------------
        # 2) direct learned model
        # ---------------------------------------
        if learned_model is not None:
            t0 = time.perf_counter()
            pred_time = predict_direct_model_time_for_case(
                case=case,
                points=points,
                model=learned_model,
                feature_names=learned_feature_names,
                target_mode=learned_target_mode,
                usv_speed=usv_speed,
                inner_width=inner_width,
                outer_width=outer_width,
            )
            t1 = time.perf_counter()

            abs_err = abs(pred_time - baseline)
            rel_err = 100.0 * abs_err / max(abs(baseline), 1e-12)

            rows.append({
                "case_id": case.case_id,
                "method": "learned_direct_prediction",
                "travel_time": float(pred_time),
                "runtime_sec": float(t1 - t0),
                "abs_error_vs_dijkstra": float(abs_err),
                "relative_error_vs_dijkstra_percent": float(rel_err),
                "path_gap_vs_dijkstra": np.nan,
                "rel_gap_vs_dijkstra_percent": np.nan,
                "start_vid": case.start_vid,
                "goal_vid": case.goal_vid,
                "start_heading_deg": case.start_heading_deg,
            })

        # ---------------------------------------
        # 3) A* Euclidean
        # ---------------------------------------
        t0 = time.perf_counter()
        res_astar_euc = shortest_path_between_vertices_astar_euclidean(
            points=points,
            cache=case.cache,
            cost_e=case.cost_e,
            theta_e_deg=case.theta_e_deg,
            start_vid=case.start_vid,
            goal_vid=case.goal_vid,
            usv_speed=usv_speed,
            th1=th1, lam1=lam1,
            th2=th2, lam2=lam2,
            use_start_heading=use_start_heading,
            start_heading_deg=case.start_heading_deg,
            termination="goal_best",
        )
        t1 = time.perf_counter()

        gap = float(res_astar_euc["best_goal"] - baseline)
        rel_gap = 100.0 * gap / max(abs(baseline), 1e-12)

        rows.append({
            "case_id": case.case_id,
            "method": "astar_euclidean",
            "travel_time": float(res_astar_euc["best_goal"]),
            "runtime_sec": float(t1 - t0),
            "abs_error_vs_dijkstra": float(abs(gap)),
            "relative_error_vs_dijkstra_percent": float(abs(rel_gap)),
            "path_gap_vs_dijkstra": float(gap),
            "rel_gap_vs_dijkstra_percent": float(rel_gap),
            "start_vid": case.start_vid,
            "goal_vid": case.goal_vid,
            "start_heading_deg": case.start_heading_deg,
        })

        # ---------------------------------------
        # 4) A* surrogate
        # ---------------------------------------
        if surrogate_model is not None:
            t0 = time.perf_counter()
            res_astar_sur = shortest_path_between_vertices_astar_surrogate(
                points=points,
                blocked_mask=case.blocked_mask,
                total_ux=case.total_ux,
                total_uy=case.total_uy,
                cache=case.cache,
                cost_e=case.cost_e,
                theta_e_deg=case.theta_e_deg,
                start_vid=case.start_vid,
                goal_vid=case.goal_vid,
                surrogate_model=surrogate_model,
                feature_names=surrogate_feature_names,
                target_mode=surrogate_target_mode,
                usv_speed=usv_speed,
                inner_width=inner_width,
                outer_width=outer_width,
                th1=th1, lam1=lam1,
                th2=th2, lam2=lam2,
                use_start_heading=use_start_heading,
                start_heading_deg=case.start_heading_deg,
                termination="goal_best",
            )
            t1 = time.perf_counter()

            gap = float(res_astar_sur["best_goal"] - baseline)
            rel_gap = 100.0 * gap / max(abs(baseline), 1e-12)

            rows.append({
                "case_id": case.case_id,
                "method": "astar_surrogate",
                "travel_time": float(res_astar_sur["best_goal"]),
                "runtime_sec": float(t1 - t0),
                "abs_error_vs_dijkstra": float(abs(gap)),
                "relative_error_vs_dijkstra_percent": float(abs(rel_gap)),
                "path_gap_vs_dijkstra": float(gap),
                "rel_gap_vs_dijkstra_percent": float(rel_gap),
                "start_vid": case.start_vid,
                "goal_vid": case.goal_vid,
                "start_heading_deg": case.start_heading_deg,
            })

    return pd.DataFrame(rows)

def summarize_benchmark_results(df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        df.groupby("method")
        .agg(
            n_cases=("case_id", "count"),

            mean_travel_time=("travel_time", "mean"),
            var_travel_time=("travel_time", "var"),
            std_travel_time=("travel_time", "std"),

            mean_runtime_sec=("runtime_sec", "mean"),
            var_runtime_sec=("runtime_sec", "var"),
            std_runtime_sec=("runtime_sec", "std"),

            mean_abs_error_vs_dijkstra=("abs_error_vs_dijkstra", "mean"),
            var_abs_error_vs_dijkstra=("abs_error_vs_dijkstra", "var"),
            std_abs_error_vs_dijkstra=("abs_error_vs_dijkstra", "std"),

            mean_rel_error_vs_dijkstra_percent=("relative_error_vs_dijkstra_percent", "mean"),
            var_rel_error_vs_dijkstra_percent=("relative_error_vs_dijkstra_percent", "var"),
            std_rel_error_vs_dijkstra_percent=("relative_error_vs_dijkstra_percent", "std"),

            mean_path_gap_vs_dijkstra=("path_gap_vs_dijkstra", "mean"),
            var_path_gap_vs_dijkstra=("path_gap_vs_dijkstra", "var"),
            std_path_gap_vs_dijkstra=("path_gap_vs_dijkstra", "std"),

            mean_rel_gap_vs_dijkstra_percent=("rel_gap_vs_dijkstra_percent", "mean"),
            var_rel_gap_vs_dijkstra_percent=("rel_gap_vs_dijkstra_percent", "var"),
            std_rel_gap_vs_dijkstra_percent=("rel_gap_vs_dijkstra_percent", "std"),
        )
        .reset_index()
    )
    return summary

In [11]:
# ---------------------------------------------------
# A. 100개 랜덤 테스트셋 생성
# ---------------------------------------------------
benchmark_cases = generate_random_benchmark_cases(
    n_cases=1000,
    points=points,
    adj_base=adj_base,
    domain=domain,
    usv_speed=2.5,
    rng_seed=20260409,

    num_obstacles=0,                # 필요하면 늘리기
    radius_range=(120.0, 200.0),
    border_margin=50.0,
    obstacle_clearance=200.0,
    min_start_goal_dist=500.0,

    th1=50.0, lam1=0.0,
    th2=60.0, lam2=np.inf,
)

print(f"#cases = {len(benchmark_cases)}")

generated 20 / 1000 cases
generated 40 / 1000 cases
generated 60 / 1000 cases
generated 80 / 1000 cases
generated 100 / 1000 cases
generated 120 / 1000 cases
generated 140 / 1000 cases
generated 160 / 1000 cases
generated 180 / 1000 cases
generated 200 / 1000 cases
generated 220 / 1000 cases
generated 240 / 1000 cases
generated 260 / 1000 cases
generated 280 / 1000 cases
generated 300 / 1000 cases
generated 320 / 1000 cases
generated 340 / 1000 cases
generated 360 / 1000 cases
generated 380 / 1000 cases
generated 400 / 1000 cases
generated 420 / 1000 cases
generated 440 / 1000 cases
generated 460 / 1000 cases
generated 480 / 1000 cases
generated 500 / 1000 cases
generated 520 / 1000 cases
generated 540 / 1000 cases
generated 560 / 1000 cases
generated 580 / 1000 cases
generated 600 / 1000 cases
generated 620 / 1000 cases
generated 640 / 1000 cases
generated 660 / 1000 cases
generated 680 / 1000 cases
generated 700 / 1000 cases
generated 720 / 1000 cases
generated 740 / 1000 cases
gener

In [20]:
# ---------------------------------------------------
# A. 저장된 residual XGB 모델 불러오기
# ---------------------------------------------------

rf_payload = joblib.load("best_residual_xgb_final.joblib")

rf_model = rf_payload["model"]
rf_model.n_jobs = 1

rf_feature_names = rf_payload["feature_names"]

# 저장할 때 target_mode도 payload에 넣었다면 이렇게
rf_target_mode = rf_payload.get("target_mode", "residual")


# ---------------------------------------------------
# B. 4가지 방식 benchmark
# ---------------------------------------------------

benchmark_df = benchmark_four_methods_on_cases(
    cases=benchmark_cases,
    points=points,
    usv_speed=2.5,

    # learned model
    learned_model=rf_model,
    learned_feature_names=rf_feature_names,
    learned_target_mode=rf_target_mode,

    # surrogate model
    surrogate_model=rf_model,
    surrogate_feature_names=rf_feature_names,
    surrogate_target_mode=rf_target_mode,

    inner_width=90.0,
    outer_width=180.0,

    th1=50.0,
    lam1=0.0,

    th2=60.0,
    lam2=np.inf,

    use_start_heading=True,
)

In [13]:
len(benchmark_df)

800

In [24]:
mean_df = benchmark_df.groupby("method").mean(numeric_only=True)
mean_df

,case_id,travel_time,runtime_sec,abs_error_vs_dijkstra,relative_error_vs_dijkstra_percent,path_gap_vs_dijkstra,rel_gap_vs_dijkstra_percent,start_vid,goal_vid,start_heading_deg
method,,,,,,,,,,
astar_euclidean,499.5,843.111218,0.194892,16.529405,2.113244,16.529405,2.113244,2116.774,2153.582,180.002878
astar_surrogate,499.5,834.169758,1.652225,7.587946,0.882657,7.587946,0.882657,2116.774,2153.582,180.002878
dijkstra,499.5,826.581812,1.875920,0.000000,0.000000,0.000000,0.000000,2116.774,2153.582,180.002878
learned_direct_prediction,499.5,828.827773,0.001460,26.790584,3.023341,NaN,NaN,2116.774,2153.582,180.002878


In [22]:
summary_df = (
    benchmark_df
    .groupby("method")
    .agg(["mean", "std"])
)

print(summary_df)

                          case_id             travel_time              \
                             mean         std        mean         std   
method                                                                  
astar_euclidean             499.5  288.819436  843.111218  358.223895   
astar_surrogate             499.5  288.819436  834.169758  358.430416   
dijkstra                    499.5  288.819436  826.581812  353.788435   
learned_direct_prediction   499.5  288.819436  828.827773  353.778032   

                          runtime_sec           abs_error_vs_dijkstra  \
                                 mean       std                  mean   
method                                                                  
astar_euclidean              0.194892  0.349209             16.529405   
astar_surrogate              1.652225  1.827578              7.587946   
dijkstra                     1.875920  1.085682              0.000000   
learned_direct_prediction    0.001460  0.000174   

In [25]:
benchmark_df.groupby("method")["relative_error_vs_dijkstra_percent"].agg(
    ["mean", "std", "median", "min", "max"]
)

,mean,std,median,min,max
method,,,,,
astar_euclidean,2.113244,2.962107,0.707687,0.000000,16.789755
astar_surrogate,0.882657,1.095686,0.504281,0.000000,8.825542
dijkstra,0.000000,0.000000,0.000000,0.000000,0.000000
learned_direct_prediction,3.023341,3.256250,1.970094,0.002404,28.853997


In [30]:
benchmark_df.to_excel("path_method.xlsx")

In [28]:

benchmark_df.groupby("method").agg('mean')

,case_id,travel_time,runtime_sec,abs_error_vs_dijkstra,relative_error_vs_dijkstra_percent,path_gap_vs_dijkstra,rel_gap_vs_dijkstra_percent,start_vid,goal_vid,start_heading_deg
method,,,,,,,,,,
astar_euclidean,499.5,843.111218,0.194892,16.529405,2.113244,16.529405,2.113244,2116.774,2153.582,180.002878
astar_surrogate,499.5,834.169758,1.652225,7.587946,0.882657,7.587946,0.882657,2116.774,2153.582,180.002878
dijkstra,499.5,826.581812,1.875920,0.000000,0.000000,0.000000,0.000000,2116.774,2153.582,180.002878
learned_direct_prediction,499.5,828.827773,0.001460,26.790584,3.023341,NaN,NaN,2116.774,2153.582,180.002878
